# ViT-MAE Base Pre-training Pipeline — DIMER E2E masked-image-modelling tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/vit-mae-pretraining-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/vit-mae-pretraining-pipeline/blob/main/tutorials/vit_mae_pretraining_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fvit--mae--base-ffcc4d?style=flat)](https://huggingface.co/facebook/vit-mae-base) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Fmae-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/mae) [![arXiv](https://img.shields.io/badge/arXiv-2111.06377-b31b1b.svg)](https://arxiv.org/abs/2111.06377)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** masked-patch reconstruction (masked image modelling), mean-pooled encoder embeddings, a linear probe on those embeddings and bounded continuation of the masked-autoencoding objective on a photograph set, using the pinned `facebook/vit-mae-base` weights

**This notebook is standalone.** It carries the repository's package (6 modules under `src/vit_mae_pipeline/`, at revision `f36b73b4ffd0`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `25b184bea5538bf5c4c852c79d221195fdd2778d` (~448 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `facebook/vit-mae-base` snapshot (a 448 MB `model.safetensors`; no pickle is opened anywhere), fetches the 360 pinned iNaturalist photographs from the project's open-data bucket (about 39 MB, each refused on any byte-size or SHA-256 mismatch), cuts them per species into 216 / 48 / 96 training, validation and test photographs, masks and reconstructs three drawn shapes through the inference contract with an input manifest and a rejection probe, measures the frozen model's masked-patch MSE over the 96 test photographs beside the mean-patch and blur fills and its linear-probe accuracy beside the majority floor, a colour nearest neighbour and a k-NN on the same features, runs a bounded continuation of the masked-autoencoding objective on the decoder and the last two encoder blocks with validation-MSE epoch selection, reconstructs and probes the held-out photographs again, re-reconstructs the drawn shapes with the adapted model, exports the adapter and the probe head as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify reconstruction and probe parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path took about four minutes of model time on the build workstation's CPU after the downloads (expect longer on a 2-vCPU hosted runtime); a CUDA runtime is used automatically when present and finishes in a few minutes.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip holding a `labels.csv` (columns `id`, `file`, `label`) beside the image files — at least eight photographs over at least two labels; the labels feed only the linear probe, the reconstruction objective never sees them. They pass through the same validation, seeded stratified split, baselines, continuation, held-out evaluation, artifact export and reload-parity cells as the iNaturalist sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`facebook/vit-mae-base` is the ViT-B/16 masked autoencoder of He, Chen, Xie, Li, Dollár and Girshick (CVPR 2022): a 12-layer, 768-wide encoder that sees only the visible 25 % of an image's 196 16×16 patches, and an 8-layer, 512-wide decoder that predicts the pixels of the hidden 75 % from the encoder's tokens plus a shared mask token — 111,907,840 parameters in total, pre-trained on ImageNet-1K for 1,600 epochs with per-patch pixel MSE on the hidden patches only, published under the **Apache-2.0** licence. It has **no user-facing task output**: the model returns a reconstruction and its own loss, and the encoder's features are what downstream fine-tuning consumes. This notebook therefore reads the model two ways — the **masked-patch MSE** of its reconstructions (the objective, in the processor's normalised pixel space, lower is better, not a quality judgement) and the **accuracy of a linear probe** on its mean-pooled patch tokens (the paper's own readout of representation quality) — and neither is calibrated or a human judgement.

What this notebook adds to inference is **continuation of the pre-training objective on a photograph set**. The dataset is real: 360 CC0-licensed, research-grade iNaturalist photographs of six common North American birds (**CC0 1.0**; four small sparrows, a junco and two finches, 60 per species, one per observer), pinned by photo id, byte size and SHA-256 and fetched from the project's open-data bucket at run time. The honest question is narrow: does a bounded continuation of masked autoencoding — the decoder and the last two encoder blocks, on 216 photographs, with the epoch chosen on the validation masked MSE — move the reconstruction error on an image-disjoint test split, against two **non-neural fills** (the **mean colour of the visible patches** and a **blur of the visible neighbours**), and does it move the probe against three **non-neural classifiers** (the **majority floor**, a **colour nearest neighbour** and a **k-NN on the same features**)? The build record's answer is that it barely does — the pre-trained model is already converged on natural photographs, and 216 more of them at a learning rate small enough not to hurt move the held-out masked MSE from 0.2281 to 0.2280 — so what the notebook demonstrates is the adaptation *contract* (bounded training, validation selection that never returns a worse epoch than the frozen model, an artifact that reloads with verified parity) and how to read the numbers around it. Nothing here is a quality claim about your photographs: it is one seeded split of one sample.

**Snapshot note:** the pinned revision ships `model.safetensors` (a 4-file manifest: model card, config, processor config and weights) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled photograph set, validate it and split it per species without leakage; mask and reconstruct drawn shapes through the public API and read a reconstruction loss correctly (the model's own objective, seeded masks, hidden-patch and visible-patch error, uncalibrated); measure the frozen model's masked-patch MSE beside two non-neural fills and its linear-probe accuracy beside three non-neural classifiers; run a bounded continuation of the masked-autoencoding objective with explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint test split both ways; re-reconstruct drawings from a different image family with the adapted model; and export a safetensors adapter with its probe head that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** image classification as a product (the probe is a readout of the features, not a classifier to ship), object detection, segmentation, generation of new images, inpainting of user-chosen regions (the mask is random, as in pre-training), full fine-tuning of the encoder for a downstream task, pre-training from scratch, normalised-pixel targets (`norm_pix_loss` is off in this checkpoint), training on photographs that are not the pinned sample or your own uploads, evaluation on ImageNet or any benchmark proper (only one seeded 360-photograph sample is scored here), and any claim that six bird species stand in for your images. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is slow but adequate: the build record measured about 4.4 s to reconstruct the 96 test photographs, 28.8 s to embed the 312 photographs and fit the probe, and 159.1 s for the 5 epochs of continuation (216 photographs per epoch through the encoder and decoder, the decoder and the last two encoder blocks training) including the per-epoch validation reconstruction, so the whole default path is about four minutes of model time on the build workstation's CPU with the snapshot and photographs already cached (a 2-vCPU hosted runtime will be several times slower); a hosted T4 finishes it in a few minutes. The pinned `torch==2.14.0` install and the 448 MB checkpoint are the large downloads of the run; the photographs add about 39 MB.
- **Knowledge:** basic Python and PIL; what a mean squared error and a PSNR are; what a masked autoencoder hides and predicts; what accuracy and macro F1 measure and why a linear probe is a readout of features rather than a product classifier.
- **Data contract:** records are `{id, image, label}` — a PIL image or a file decodable by Pillow with sides between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px, and a label of at most 64 plain characters (used by the probe only). Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 8..20,000 records over 2..100 labels; splitting is stratified per label after pixel-digest de-duplication so no photograph lands in two splits. Every image is resized to 224×224 by the processor. BYOD accepts one zip of images plus a `labels.csv` in that shape.
- **Validation is structural, not semantic:** every image is opened and decoded and every label checked, but nothing checks that a label is right — a mislabelled set is probed without complaint, and the reconstruction objective never reads the labels at all.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path fetches 360 JPEG/PNG files from `https://inaturalist-open-data.s3.amazonaws.com/photos/<id>/medium.<ext>` (about 39 MB in total), each pinned by byte size and SHA-256 in the carried `samples.py` and refused on any mismatch; every photograph's iNaturalist observation page and observer login are kept in its record. Each photograph carries the CC0 1.0 licence its observer chose (nothing is committed to the repository).
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/vit-mae-base` snapshot (~448 MB in total) at revision `25b184bea553…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'safetensors==0.8.0',
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
]
NOTEBOOK_SOURCE = {
    'repository': 'vit-mae-pretraining-pipeline',
    'repository_revision': 'f36b73b4ffd07e9c4dd68c79c6d42c01c9392b05',
    'embedded_module': 'src/vit_mae_pipeline/config.py',
    'embedded_modules': ['src/vit_mae_pipeline/config.py', 'src/vit_mae_pipeline/metrics.py', 'src/vit_mae_pipeline/model.py', 'src/vit_mae_pipeline/provenance.py', 'src/vit_mae_pipeline/samples.py', 'src/vit_mae_pipeline/pipeline.py'],
    'module_sha256': '7b2d3b48b694c7c9c6427c4c4a09f802a14408a44d1ac816769a54d88c08a303',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/vit_mae_pipeline/` @ `f36b73b4ffd0`)

The next 6 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (2 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/6:** `src/vit_mae_pipeline/config.py`

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

MODEL_ID = "facebook/vit-mae-base"
MODEL_REVISION = "25b184bea5538bf5c4c852c79d221195fdd2778d"
MODEL_FILENAME = "model.safetensors"
MODEL_SHA256 = "479dcef4bd5df06259399027b789f21e9d9a1b79f37155a64176d55bc26fdae8"
MODEL_SIZE_BYTES = 447_670_680
MODEL_LICENSE = "Apache-2.0"

DEFAULT_MODEL_KEY = "vit-mae-base"
UNSAFE_WEIGHT_EXTENSIONS = (
    ".bin",
    ".pt",
    ".pth",
    ".ckpt",
    ".pkl",
    ".pickle",
    ".h5",
    ".msgpack",
)

ALLOWED_CHECKPOINT_FILES = (
    "config.json",
    MODEL_FILENAME,
    "preprocessor_config.json",
)

# The checkpoint's own geometry (config.json at the pinned revision).
IMAGE_SIZE = 224
PATCH_SIZE = 16
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 196
HIDDEN_SIZE = 768
ENCODER_LAYERS = 12
DECODER_HIDDEN_SIZE = 512
DECODER_LAYERS = 8
DEFAULT_MASK_RATIO = 0.75  # config.json mask_ratio: 147 of the 196 patches are hidden from the encoder
NORM_PIX_LOSS = False  # config.json norm_pix_loss: the loss is a raw-pixel MSE on the masked patches

**Module 2/6:** `src/vit_mae_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level measures for the two things this pipeline can be read on, in numpy.

**Reconstruction** (what a masked autoencoder is trained for): the mean squared error between the decoder's
prediction and the true pixels of the patches that were hidden from the encoder, averaged over the hidden
patches of an image and then over images — the checkpoint's own training loss (`norm_pix_loss` false: raw
pixels in the processor's normalised space). Two non-neural references a fine-tuned model must beat: the
**mean-patch fill** (every hidden patch predicted as the mean colour of the visible patches — a decoder that
knows only the image's average) and the **blur fill** (every hidden patch predicted as the mean of its visible
neighbours in the 14 x 14 patch grid — the cheapest spatial prior).

**Linear-probe classification** (how the fleet reads a self-supervised encoder): accuracy and macro F1 of a
linear head on frozen embeddings, with a per-class breakdown, against the **majority floor** and a **colour
nearest neighbour** (the label of the training photograph whose 3 x 3 mean-colour grid is closest — a
classifier that knows the image through 27 numbers) and a **k-nearest-neighbour** on the same embeddings.
"""
# ruff: noqa: E501  -- fleet metrics module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import math
from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np
from PIL import Image

COLOUR_GRID = 3
METRIC_DEFINITIONS = {
    "accuracy": "fraction of photographs whose highest-scoring class is the gold label; in 0..1",
    "macro_f1": "unweighted mean over classes of the F1 of predicting that class; in 0..1",
}
RECONSTRUCTION_DEFINITIONS = {
    "masked_mse": (
        "mean over images of the mean squared error between predicted and true pixels over the patches hidden "
        "from the encoder, in the processor's normalised pixel space (ImageNet mean / std); lower is better"
    ),
    "masked_mse_visible": "the same error over the patches the encoder saw (the decoder also predicts those); lower is better",
    "mask_ratio": "fraction of the patches hidden from the encoder (the seeded random mask)",
    "psnr_masked": "peak signal-to-noise ratio of the hidden patches in 0..1 pixel space, dB; higher is better",
}


# ---------------------------------------------------------------------------------- reconstruction


def masked_mse(prediction: np.ndarray, target: np.ndarray, mask: np.ndarray) -> float:
    """MSE over the patches with mask == 1. `prediction` / `target` are `[patches, values]`, `mask` `[patches]`."""
    pred = np.asarray(prediction, dtype=np.float64)
    true = np.asarray(target, dtype=np.float64)
    m = np.asarray(mask, dtype=np.float64)
    if pred.shape != true.shape or pred.ndim != 2 or m.shape != (pred.shape[0],):
        raise ValueError("prediction and target must be [patches, values] with one mask entry per patch")
    if m.sum() <= 0:
        raise ValueError("at least one patch must be masked")
    per_patch = ((pred - true) ** 2).mean(axis=1)
    return float((per_patch * m).sum() / m.sum())


def psnr_from_mse(mse_normalised: float, *, std: Sequence[float] = (0.229, 0.224, 0.225)) -> float:
    """PSNR (dB) in 0..1 pixel space from an MSE measured in ImageNet-normalised space (per-channel std applied
    as the mean squared std; an approximation stated as such)."""
    scale = float(np.mean(np.square(np.asarray(std, dtype=np.float64))))
    mse_pixels = max(mse_normalised * scale, 1e-12)
    return float(10.0 * math.log10(1.0 / mse_pixels))


def reconstruction_metrics(rows: Sequence[Mapping[str, Any]], *, mask_ratio: float) -> dict[str, Any]:
    """Aggregate per-image rows `{id, masked_mse, visible_mse, ...}` into the corpus reading."""
    if not rows:
        raise ValueError("no images to score")
    masked = np.asarray([r["masked_mse"] for r in rows], dtype=np.float64)
    visible = np.asarray([r["visible_mse"] for r in rows], dtype=np.float64)
    if not (np.all(np.isfinite(masked)) and np.all(np.isfinite(visible))):
        raise ValueError("reconstruction errors must be finite")
    return {
        "n": int(len(rows)),
        "masked_mse": float(masked.mean()),
        "masked_mse_median": float(np.median(masked)),
        "masked_mse_visible": float(visible.mean()),
        "psnr_masked": psnr_from_mse(float(masked.mean())),
        "mask_ratio": float(mask_ratio),
        "definitions": dict(RECONSTRUCTION_DEFINITIONS),
    }


def mean_patch_fill(patches: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Predict every hidden patch as the mean over the visible patches (per value position averaged to one
    colour per channel): the decoder that knows only the image's average."""
    p = np.asarray(patches, dtype=np.float64)
    m = np.asarray(mask, dtype=bool)
    visible = p[~m]
    if not len(visible):
        raise ValueError("no visible patches")
    values = visible.shape[1]
    channels = 3
    per_channel = visible.reshape(-1, values // channels, channels).mean(axis=(0, 1))
    fill = np.tile(per_channel, values // channels)
    out = p.copy()
    out[m] = fill
    return out


def blur_fill(patches: np.ndarray, mask: np.ndarray, *, grid: int) -> np.ndarray:
    """Predict every hidden patch as the mean colour of its visible 8-neighbours in the patch grid (falling back
    to the mean-patch fill where a patch has no visible neighbour)."""
    p = np.asarray(patches, dtype=np.float64)
    m = np.asarray(mask, dtype=bool)
    if p.shape[0] != grid * grid:
        raise ValueError("patch count must equal grid * grid")
    values = p.shape[1]
    channels = 3
    colours = p.reshape(grid * grid, values // channels, channels).mean(axis=1)  # one colour per patch
    fallback = colours[~m].mean(axis=0) if (~m).any() else np.zeros(channels)
    out = p.copy()
    for index in np.flatnonzero(m):
        row, col = divmod(int(index), grid)
        neighbours = []
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == dc == 0:
                    continue
                r2, c2 = row + dr, col + dc
                if 0 <= r2 < grid and 0 <= c2 < grid and not m[r2 * grid + c2]:
                    neighbours.append(colours[r2 * grid + c2])
        colour = np.mean(neighbours, axis=0) if neighbours else fallback
        out[index] = np.tile(colour, values // channels)
    return out


# ---------------------------------------------------------------------------------- classification


def _per_class(pred: np.ndarray, gold: np.ndarray, n_classes: int) -> dict[str, Any]:
    out = {}
    for c in range(n_classes):
        tp = int(np.sum((pred == c) & (gold == c)))
        fp = int(np.sum((pred == c) & (gold != c)))
        fn = int(np.sum((pred != c) & (gold == c)))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        out[c] = {"support": int(np.sum(gold == c)), "precision": precision, "recall": recall, "f1": f1}
    return out


def classification_metrics(scores: Any, gold: Sequence[int], classes: Sequence[str]) -> dict[str, Any]:
    """Accuracy, macro F1 and a per-class breakdown for an `[image][class]` score grid."""
    grid = np.asarray(scores, dtype=np.float64)
    gold_arr = np.asarray(gold, dtype=int)
    if grid.ndim != 2 or grid.shape != (len(gold_arr), len(classes)):
        raise ValueError(f"scores must be an [images x classes] grid, got {grid.shape}")
    if not len(gold_arr) or gold_arr.min() < 0 or gold_arr.max() >= len(classes):
        raise ValueError("gold must hold class indices for at least one photograph")
    if not np.all(np.isfinite(grid)):
        raise ValueError("scores must be finite")
    pred = grid.argmax(axis=1)
    per = _per_class(pred, gold_arr, len(classes))
    return {
        "n": int(len(gold_arr)),
        "accuracy": float(np.mean(pred == gold_arr)),
        "macro_f1": float(np.mean([per[c]["f1"] for c in range(len(classes))])),
        "per_class": {classes[c]: per[c] for c in range(len(classes))},
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _scores_from_predictions(pred: Sequence[int], n_classes: int) -> np.ndarray:
    grid = np.zeros((len(pred), n_classes))
    for i, c in enumerate(pred):
        grid[i, c] = 1.0
    return grid


def majority_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], classes: Sequence[str]
) -> dict[str, Any]:
    """The most frequent training label for every photograph."""
    if not train:
        raise ValueError("the majority baseline needs training records")
    counts = Counter(str(r["label"]) for r in train)
    label = max(sorted(counts), key=counts.get)
    index = list(classes).index(label)
    gold = [list(classes).index(str(r["label"])) for r in records]
    result = classification_metrics(_scores_from_predictions([index] * len(records), len(classes)), gold, classes)
    result["baseline"] = f"majority floor ({label!r} for every photograph)"
    return result


def colour_signature(image: str | Image.Image, *, grid: int = COLOUR_GRID) -> list[float]:
    """Mean RGB of each cell of a `grid` x `grid` partition of the image, in 0..1 (27 numbers by default)."""
    handle = image if isinstance(image, Image.Image) else Image.open(image)
    small = handle.convert("RGB").resize((grid * 8, grid * 8), Image.BILINEAR)
    pixels = list(small.getdata())
    out: list[float] = []
    for row in range(grid):
        for col in range(grid):
            cell = [pixels[(row * 8 + y) * grid * 8 + col * 8 + x] for y in range(8) for x in range(8)]
            out.extend(sum(p[channel] for p in cell) / (64 * 255.0) for channel in range(3))
    return out


def _distance(a: Sequence[float], b: Sequence[float]) -> float:
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b, strict=True)))


def colour_neighbour_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], classes: Sequence[str]
) -> dict[str, Any]:
    """Label every photograph with the label of the training photograph whose colour signature is closest;
    the score grid is the negative distance to the nearest training photograph of each class."""
    if not train:
        raise ValueError("the colour-neighbour baseline needs training records")
    class_list = list(classes)
    signatures = [(colour_signature(r["image"]), class_list.index(str(r["label"]))) for r in train]
    grid = np.zeros((len(records), len(class_list)))
    for i, record in enumerate(records):
        query = colour_signature(record["image"])
        best = [math.inf] * len(class_list)
        for sig, c in signatures:
            best[c] = min(best[c], _distance(sig, query))
        grid[i] = [-d for d in best]
    gold = [class_list.index(str(r["label"])) for r in records]
    result = classification_metrics(grid, gold, class_list)
    result["baseline"] = f"colour nearest neighbour ({len(train)} training photographs)"
    return result


def knn_scores(train_features: np.ndarray, train_gold: Sequence[int], features: np.ndarray, n_classes: int, *, k: int = 5) -> np.ndarray:
    """Cosine k-NN vote on L2-normalised features: the score grid is the summed similarity of the k nearest
    training features per class."""
    a = np.asarray(train_features, dtype=np.float64)
    b = np.asarray(features, dtype=np.float64)
    gold = np.asarray(train_gold, dtype=int)
    if a.ndim != 2 or b.ndim != 2 or a.shape[1] != b.shape[1] or len(gold) != len(a):
        raise ValueError("feature matrices must be [images x dim] with one training label per row")
    sims = b @ a.T
    k = min(k, len(a))
    grid = np.zeros((len(b), n_classes))
    for i in range(len(b)):
        top = np.argsort(-sims[i])[:k]
        for j in top:
            grid[i, gold[j]] += sims[i, j]
    return grid

**Module 3/6:** `src/vit_mae_pipeline/model.py` (carried verbatim; see the note above)

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

import hashlib
import json
import os
from collections.abc import Callable
from pathlib import Path
from typing import Any

import torch
from huggingface_hub import snapshot_download
from transformers import AutoImageProcessor, ViTMAEForPreTraining

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

MANIFEST_NAME = "dimer-base-manifest.json"
#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files live in a repository-
#: local snapshot directory named by the model key and described by the committed manifest; a
#: standalone notebook carries that manifest inline and stages/verifies a working-directory copy.
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / DEFAULT_MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory snapshot, no repository checkout


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_checkpoint(
    snapshot_path: str | Path,
    *,
    require_configs: bool = False,
    return_manifest_verified: bool = False,
) -> Path | tuple[Path, bool]:
    root = Path(snapshot_path)
    if not root.is_dir():
        raise RuntimeError(f"Checkpoint directory does not exist: {root}")

    weight_path = root / MODEL_FILENAME
    if not weight_path.is_file():
        raise RuntimeError(f"Pinned checkpoint is missing {MODEL_FILENAME}")

    unsafe = sorted(
        p.name
        for p in root.iterdir()
        if p.is_file() and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS
    )
    if unsafe:
        raise RuntimeError(f"Refusing unsafe weight files: {unsafe}")

    manifest_path = root / MANIFEST_NAME
    manifest_verified = False

    if manifest_path.is_file():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc

        files = manifest.get("files") or []
        if not files:
            raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")

        for entry in files:
            rel_path = entry.get("path")
            if not rel_path:
                continue
            target = root / rel_path
            if not target.is_file():
                raise RuntimeError(f"Manifest file missing: {rel_path}")
            exp_bytes = entry.get("bytes")
            if exp_bytes is not None and target.stat().st_size != exp_bytes:
                raise RuntimeError(
                    f"Size mismatch for {rel_path}: {target.stat().st_size} != {exp_bytes}"
                )
            exp_sha = entry.get("sha256")
            if exp_sha is not None and _sha256(target) != exp_sha:
                raise RuntimeError(f"SHA-256 mismatch for {rel_path}")

        manifest_verified = True

    size = weight_path.stat().st_size
    if size != MODEL_SIZE_BYTES:
        raise RuntimeError(f"Unexpected {MODEL_FILENAME} size: {size}; expected {MODEL_SIZE_BYTES}")

    digest = _sha256(weight_path)
    if digest != MODEL_SHA256:
        raise RuntimeError(
            f"Unexpected {MODEL_FILENAME} SHA-256: {digest}; expected {MODEL_SHA256}"
        )

    if require_configs:
        for req in ("config.json", "preprocessor_config.json"):
            if not (root / req).is_file():
                raise RuntimeError(f"Missing required configuration file: {req}")

    if return_manifest_verified:
        return root, manifest_verified
    return root


def _read_manifest(root: Path) -> dict[str, Any]:
    """Load and identity-check ``<root>/dimer-base-manifest.json``."""
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except ValueError as exc:
        raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing"
        )
    if not manifest.get("files"):
        raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Manifest-driven verification of a fleet snapshot directory; raise on the first mismatch.

    The identity in the manifest must be the pinned one; every manifest entry is then size- and
    SHA-256-checked by :func:`verify_checkpoint` (the existing verifier, which also asserts the
    weight file's pinned digest and byte count and refuses unsafe formats). Returns
    ``{"path": ..., **manifest}``.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    _, manifest_verified = verify_checkpoint(
        root, require_configs=True, return_manifest_verified=True
    )
    if not manifest_verified:
        raise RuntimeError(f"manifest at {root} was not verified")  # pragma: no cover
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and
    the small files but git-ignores the weights). Returns the relative paths fetched;
    :func:`verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def resolve_weights_path(
    weights_path: str | Path | None = None,
    cache_dir: str | Path | None = None,
) -> tuple[Path, str]:
    """Resolve weights path with precedence:

    1. Explicit argument `weights_path` -> 'explicit_path'
    2. Environment variable `VIT_MAE_WEIGHTS_DIR` -> 'env_var'
    3. Source checkout convention `weights/vit-mae-base` -> 'repo_offline'
       (only if pyproject.toml exists at repo root and weights/ contains model.safetensors)
    4. Hugging Face Hub snapshot download -> 'hf_hub'
    """
    if weights_path is not None:
        return Path(weights_path), "explicit_path"

    env_dir = os.environ.get("VIT_MAE_WEIGHTS_DIR")
    if env_dir:
        return Path(env_dir), "env_var"

    repo_root = Path.cwd()  # standalone rewrite (build_notebook.py): no repository checkout to resolve
    if (repo_root / "pyproject.toml").is_file():
        repo_weights = repo_root / "weights" / DEFAULT_MODEL_KEY
        if (repo_weights / MODEL_FILENAME).is_file():
            return repo_weights, "repo_offline"

    hub_path = Path(
        snapshot_download(
            repo_id=MODEL_ID,
            revision=MODEL_REVISION,
            allow_patterns=list(ALLOWED_CHECKPOINT_FILES),
            cache_dir=str(cache_dir) if cache_dir is not None else None,
        )
    )
    return hub_path, "hf_hub"


_resolve_weights_path = resolve_weights_path


def _resolve_device(device: str | torch.device | None) -> torch.device:
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_components(
    *,
    device: str | torch.device | None = None,
    cache_dir: str | Path | None = None,
    weights_path: str | Path | None = None,
    return_metadata: bool = False,
) -> tuple[Any, Any, torch.device, Path] | tuple[Any, Any, torch.device, Path, dict[str, Any]]:
    """Acquire, verify, and load the one supported ViT-MAE checkpoint (encoder + decoder, ViTMAEForPreTraining)."""

    candidate_path, source = resolve_weights_path(
        weights_path=weights_path,
        cache_dir=cache_dir,
    )

    verified, manifest_verified = verify_checkpoint(
        candidate_path,
        require_configs=True,
        return_manifest_verified=True,
    )
    target_device = _resolve_device(device)

    processor = AutoImageProcessor.from_pretrained(
        verified,
        local_files_only=True,
        trust_remote_code=False,
    )
    model = ViTMAEForPreTraining.from_pretrained(
        verified,
        local_files_only=True,
        trust_remote_code=False,
        use_safetensors=True,
    )
    model = model.eval().to(target_device)

    weight_file = verified / MODEL_FILENAME
    metadata: dict[str, Any] = {
        "checkpoint_path": verified,
        "checkpoint_source": source,
        "manifest_verified": manifest_verified,
        "weight_sha256": _sha256(weight_file),
        "weight_size_bytes": weight_file.stat().st_size,
        "device": str(target_device),
    }

    if return_metadata:
        return model, processor, target_device, verified, metadata
    return model, processor, target_device, verified

**Module 4/6:** `src/vit_mae_pipeline/provenance.py` (carried verbatim; see the note above)

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

import json
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

_RUNTIME_PACKAGES = ("torch", "transformers", "safetensors", "numpy", "pillow", "huggingface-hub")


def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def build_provenance(
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> dict[str, Any]:
    checkpoint_source = None
    manifest_verified = False
    weight_sha256 = MODEL_SHA256
    weight_size = MODEL_SIZE_BYTES
    device_str = None
    resolved_checkpoint_path = None
    adapter: dict[str, Any] | None = None

    if pipeline is not None:
        if getattr(pipeline, "checkpoint_path", None) is not None:
            resolved_checkpoint_path = str(pipeline.checkpoint_path)
        checkpoint_source = getattr(pipeline, "checkpoint_source", None)
        manifest_verified = bool(getattr(pipeline, "manifest_verified", False))
        if getattr(pipeline, "weight_sha256", None):
            weight_sha256 = pipeline.weight_sha256
        if getattr(pipeline, "weight_size_bytes", None):
            weight_size = pipeline.weight_size_bytes
        if getattr(pipeline, "device", None) is not None:
            device_str = str(pipeline.device)
        if getattr(pipeline, "adapter", None):
            skip = ("history", "trainable_names")
            adapter = {k: v for k, v in pipeline.adapter.items() if k not in skip}

    if checkpoint_path is not None:
        resolved_checkpoint_path = str(checkpoint_path)
        if checkpoint_source is None:
            checkpoint_source = "explicit_path"

    model_record: dict[str, Any] = {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "license": MODEL_LICENSE,
        "weight_file": MODEL_FILENAME,
        "weight_sha256": weight_sha256,
        "weight_size_bytes": weight_size,
        "weight_format": "safetensors",
    }
    if checkpoint_source is not None:
        model_record["checkpoint_source"] = checkpoint_source
    if resolved_checkpoint_path is not None:
        model_record["checkpoint_path"] = resolved_checkpoint_path
    if pipeline is not None or checkpoint_path is not None:
        model_record["manifest_verified"] = manifest_verified

    inference_record: dict[str, Any] = {
        "reconstruction": (
            "masked autoencoding: a seeded random subset of the patches is hidden from the encoder, the decoder "
            "predicts every patch's pixels, the loss is the mean squared error on the hidden patches only"
        ),
        "default_mask_ratio": DEFAULT_MASK_RATIO,
        "norm_pix_loss": NORM_PIX_LOSS,
        "loss_semantics": "raw-pixel MSE over the normalised (ImageNet mean / std) hidden patches; lower is better; not calibrated",
        "embedding": "mean of the encoder's patch tokens after the final LayerNorm with no patch hidden (mask ratio 0), L2-normalised",
        "embedding_dim": 768,
    }
    if device_str is not None:
        inference_record["device"] = device_str

    provenance: dict[str, Any] = {
        "schema_version": 1,
        "model": model_record,
        "processor": {
            "input_resolution": [IMAGE_SIZE, IMAGE_SIZE],
            "patch_size": PATCH_SIZE,
            "num_patches": NUM_PATCHES,
            "normalization": "ImageNet mean / std (preprocessor_config.json)",
        },
        "inference": inference_record,
        "adapter": adapter,
    }
    if include_runtime:
        provenance["runtime"] = {
            "python": platform.python_version(),
            "implementation": platform.python_implementation(),
            "platform": sys.platform,
            "packages": {name: _package_version(name) for name in _RUNTIME_PACKAGES},
        }
    return provenance


def write_provenance(
    path: str | Path,
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> Path:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            build_provenance(pipeline=pipeline, checkpoint_path=checkpoint_path, include_runtime=include_runtime),
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    return target

**Module 5/6:** `src/vit_mae_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Image dataset contract for continuing the masked-autoencoding pretraining and reading the encoder through a
linear probe: the pinned iNaturalist bird sample, validation, seeded splitting, BYOD loaders and CSV export.

The default dataset is **real** and out of the checkpoint's ImageNet-1k pretraining domain in the way that
matters here — fine-grained wildlife: 360 CC0-licensed, research-grade iNaturalist photographs of six common
North American birds (60 per species, one per observer per species; four small sparrows, a junco and two
finches), the fleet's shared sample, pinned here by photo id, byte size and SHA-256 of the served `medium`
JPEG. Every file is fetched from the iNaturalist open-data bucket at run time and refused on any byte-size or
SHA-256 mismatch; the repository redistributes none of the photographs. Each record keeps the observation id
and observer login so every image is traceable to its public observation page.

A record is ``{id, image, label}``: a PIL image (or a path to one) and the species key of its gold label —
the label is used by the linear probe only; the reconstruction contract and `adapt` never read it, and
`validate_dataset(..., require_labels=False)` accepts unlabelled ``{id, image}`` records for them. `SPECIES`
maps the key to the scientific and common names for provenance.
"""
# ruff: noqa: E501  -- fleet dataset module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import MODEL_ID` removed — names are kernel globals defined by the carried modules

MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work

CORPUS_NAME = "iNaturalist CC0 bird photographs (six species)"
CORPUS_RELEASE = (
    "iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (60 per species)"
)
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = (
    "CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)"
)
CORPUS_BYTES = 39_223_447
DEFAULT_CACHE_DIR = Path("weights") / "inat-birds"
SPECIES: dict[str, tuple[str, str]] = {
    "song_sparrow": ("Melospiza melodia", "Song Sparrow"),
    "chipping_sparrow": ("Spizella passerina", "Chipping Sparrow"),
    "white_throated_sparrow": ("Zonotrichia albicollis", "White-throated Sparrow"),
    "dark_eyed_junco": ("Junco hyemalis", "Dark-eyed Junco"),
    "house_finch": ("Haemorhous mexicanus", "House Finch"),
    "american_goldfinch": ("Spinus tristis", "American Goldfinch"),
}
# (id, label, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served
#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension
#  (jpg or jpeg); the digest pins the served bytes
SAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (
    (
        "song_sparrow-00",
        "song_sparrow",
        129376982,
        79016324,
        "andywilson",
        43427,
        "7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d",
        "jpg",
    ),
    (
        "song_sparrow-01",
        "song_sparrow",
        480991086,
        267636534,
        "lyneisfilm",
        162073,
        "11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec",
        "jpg",
    ),
    (
        "song_sparrow-02",
        "song_sparrow",
        546060381,
        302980489,
        "swpollinators",
        27899,
        "4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6",
        "jpg",
    ),
    (
        "song_sparrow-03",
        "song_sparrow",
        308625896,
        177450028,
        "radrat",
        70961,
        "1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5",
        "jpeg",
    ),
    (
        "song_sparrow-04",
        "song_sparrow",
        494793016,
        275349085,
        "k-simpkins",
        58410,
        "255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e",
        "jpg",
    ),
    (
        "song_sparrow-05",
        "song_sparrow",
        674054489,
        369029444,
        "ben142",
        220573,
        "1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123",
        "jpg",
    ),
    (
        "song_sparrow-06",
        "song_sparrow",
        339623726,
        193339933,
        "rawcomposition",
        25012,
        "d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b",
        "jpg",
    ),
    (
        "song_sparrow-07",
        "song_sparrow",
        181658744,
        107953669,
        "gcart043",
        98482,
        "a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4",
        "jpeg",
    ),
    (
        "song_sparrow-08",
        "song_sparrow",
        222768957,
        130949329,
        "davidfbird",
        110773,
        "0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff",
        "jpg",
    ),
    (
        "song_sparrow-09",
        "song_sparrow",
        148994242,
        90171417,
        "glennberry",
        102702,
        "64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa",
        "jpg",
    ),
    (
        "song_sparrow-10",
        "song_sparrow",
        637315932,
        349374463,
        "sooji",
        136572,
        "a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033",
        "jpg",
    ),
    (
        "song_sparrow-11",
        "song_sparrow",
        471146686,
        262252507,
        "jeanpaulboerekamps",
        98601,
        "4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b",
        "jpg",
    ),
    (
        "song_sparrow-12",
        "song_sparrow",
        640811426,
        351179648,
        "erikschiff",
        107695,
        "27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b",
        "jpg",
    ),
    (
        "song_sparrow-13",
        "song_sparrow",
        123859010,
        75689904,
        "w_mark_c",
        190819,
        "6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33",
        "jpg",
    ),
    (
        "song_sparrow-14",
        "song_sparrow",
        108520869,
        67204020,
        "dugald",
        52496,
        "e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8",
        "jpg",
    ),
    (
        "song_sparrow-15",
        "song_sparrow",
        63537800,
        40010230,
        "nathanael15",
        51441,
        "5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8",
        "jpg",
    ),
    (
        "song_sparrow-16",
        "song_sparrow",
        435251292,
        244042351,
        "carterdorscht",
        166662,
        "d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a",
        "jpeg",
    ),
    (
        "song_sparrow-17",
        "song_sparrow",
        120710033,
        73898230,
        "tys_rbg",
        126820,
        "515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41",
        "jpg",
    ),
    (
        "song_sparrow-18",
        "song_sparrow",
        349361283,
        198243065,
        "sean579",
        42820,
        "5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074",
        "jpg",
    ),
    (
        "song_sparrow-19",
        "song_sparrow",
        393408927,
        222067370,
        "irenemacaulay_",
        101681,
        "22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c",
        "jpg",
    ),
    (
        "song_sparrow-20",
        "song_sparrow",
        614258628,
        337847585,
        "joy4birds",
        70552,
        "aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67",
        "jpg",
    ),
    (
        "song_sparrow-21",
        "song_sparrow",
        608802621,
        335154467,
        "jamesadney",
        112564,
        "6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513",
        "jpg",
    ),
    (
        "song_sparrow-22",
        "song_sparrow",
        634100352,
        347744524,
        "zorthesosen",
        214044,
        "6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd",
        "jpg",
    ),
    (
        "song_sparrow-23",
        "song_sparrow",
        50065474,
        31954532,
        "truthseqr",
        82521,
        "4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15",
        "jpeg",
    ),
    (
        "song_sparrow-24",
        "song_sparrow",
        8656044,
        6803564,
        "glmory",
        297964,
        "e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5",
        "jpg",
    ),
    (
        "song_sparrow-25",
        "song_sparrow",
        131051149,
        79988590,
        "funvill",
        72572,
        "0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894",
        "jpeg",
    ),
    (
        "song_sparrow-26",
        "song_sparrow",
        12923156,
        9491600,
        "gambolingquail",
        69559,
        "6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d",
        "jpg",
    ),
    (
        "song_sparrow-27",
        "song_sparrow",
        12077500,
        8959545,
        "reuvenm",
        74010,
        "3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593",
        "jpeg",
    ),
    (
        "song_sparrow-28",
        "song_sparrow",
        132730299,
        80955309,
        "steph123456",
        154309,
        "5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270",
        "jpeg",
    ),
    (
        "song_sparrow-29",
        "song_sparrow",
        124840110,
        76316566,
        "terrimewbornagain",
        81046,
        "2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6",
        "jpeg",
    ),
    (
        "song_sparrow-30",
        "song_sparrow",
        130188079,
        79482300,
        "fake_id",
        180462,
        "1d9ad7b45536b85d46fa0fd9a29d4a8ae828bc223836796a3eb9df1fe8fe4fc4",
        "jpeg",
    ),
    (
        "song_sparrow-31",
        "song_sparrow",
        136405967,
        83075831,
        "tom_lazar",
        178554,
        "f86c3c5899d20ac8fb73afb76d45666c610245636d7285a6d517b0081b9d6cc9",
        "jpeg",
    ),
    (
        "song_sparrow-32",
        "song_sparrow",
        118885385,
        72867352,
        "ellyne",
        105156,
        "aab53fb1e7c12adc697711215778c81983f51af83e20bfffe841b80c1d144673",
        "jpeg",
    ),
    (
        "song_sparrow-33",
        "song_sparrow",
        676467457,
        370293984,
        "sholsenbeck",
        59104,
        "b7c230ca942cedc1c53f42902e1f70bc2d09497e57d2e9f31dfe72a5f6c84d16",
        "jpg",
    ),
    (
        "song_sparrow-34",
        "song_sparrow",
        688316087,
        376461614,
        "doublecritch",
        109416,
        "7f11c69df733f99d6e8663e91d807327f8cc57f404049fdbe95e2fd1ce1066e1",
        "jpg",
    ),
    (
        "song_sparrow-35",
        "song_sparrow",
        691785499,
        378254663,
        "karuquebec",
        144522,
        "9862b079bcf496aa8d9ce9e716e65b03a3d9a70a5c2077a223630cf9dcc6d5cc",
        "jpg",
    ),
    (
        "song_sparrow-36",
        "song_sparrow",
        691561091,
        378140427,
        "vaughnshirey",
        87827,
        "a2c197f91c0543ec7bd3a9aac4867a34417b2eb2444bf3062060f9f1e3580045",
        "jpg",
    ),
    (
        "song_sparrow-37",
        "song_sparrow",
        582182748,
        321880490,
        "dougbrown",
        44795,
        "465ce1214c40efab6afdddfbfd34fc5076e5640d130b0442e8e8ac2731ec8a1f",
        "jpg",
    ),
    (
        "song_sparrow-38",
        "song_sparrow",
        704199014,
        384692464,
        "enspring",
        85269,
        "1aa25fd1ed376b9245127432925832cf9253e5d156f4f7d35298eff9a8002043",
        "jpg",
    ),
    (
        "song_sparrow-39",
        "song_sparrow",
        583754170,
        322666295,
        "rociherrera",
        97743,
        "5e093fc6d59b13bbd7ee0889613cef115ef7bb14432634d0af753a8124951737",
        "jpg",
    ),
    (
        "song_sparrow-40",
        "song_sparrow",
        590270305,
        325961498,
        "damienxw",
        133375,
        "a12a152c94ef9c27e5f6dc557e74bf57f6c9ca3015a5e28ee9c30a27ed0e7e19",
        "jpg",
    ),
    (
        "song_sparrow-41",
        "song_sparrow",
        423534602,
        237940479,
        "don54",
        25229,
        "69b0f0dfb05e8219700e3878c9b7e4a2d43d0a2df3905414630913397a565dea",
        "jpeg",
    ),
    (
        "song_sparrow-42",
        "song_sparrow",
        418537114,
        235351206,
        "memoosborne",
        95540,
        "3432cf5a95939b898989c5a962c66b2d71b92b17195d5e695020a776361af3fa",
        "jpeg",
    ),
    (
        "song_sparrow-43",
        "song_sparrow",
        280724703,
        162358184,
        "shirleymorrison",
        43015,
        "f64af9b3fc4559f251ba0f1e95bf8889bfa629123586532adeb2b81e4c2871b2",
        "jpg",
    ),
    (
        "song_sparrow-44",
        "song_sparrow",
        641009256,
        351278414,
        "robinlanark",
        49640,
        "e4af33fd4a504a876b6e7a111a5024cf4197fc888c90f2039df63e9be13d5f06",
        "jpg",
    ),
    (
        "song_sparrow-45",
        "song_sparrow",
        531314531,
        295151228,
        "steveplumb",
        128710,
        "8806861909c0833b29f43b10e9428e104deaccb99d41b42039b76f7da0ae4600",
        "jpg",
    ),
    (
        "song_sparrow-46",
        "song_sparrow",
        114487462,
        70425556,
        "leahmfulton",
        50091,
        "d0b3c9408b72514f8485b19f61116b94be002be99368db96e126d625a17bd343",
        "jpg",
    ),
    (
        "song_sparrow-47",
        "song_sparrow",
        118804811,
        72823547,
        "heibudas",
        191643,
        "8dbe5a56d5fea83b6c25257aa0f9e552ea971f1e0311f583691d24fc9668556f",
        "jpeg",
    ),
    (
        "song_sparrow-48",
        "song_sparrow",
        118498513,
        72671334,
        "seanwashington1",
        80800,
        "940762ff238d94815922d06051ca00fe4a025279b1da08921a44733084adcdb6",
        "jpeg",
    ),
    (
        "song_sparrow-49",
        "song_sparrow",
        119696805,
        73327300,
        "rambryum",
        68983,
        "65f7007d6b90654b77b7f4ec0557ff2d0f310c572368cb3492dd69f9d082e90d",
        "jpeg",
    ),
    (
        "song_sparrow-50",
        "song_sparrow",
        4251416,
        3670620,
        "swells",
        49271,
        "8c53fa5c02e603227117293720d269c59a13555abd537a83640210805fbd844d",
        "JPG",
    ),
    (
        "song_sparrow-51",
        "song_sparrow",
        303954368,
        174929167,
        "dianeclark6280",
        157478,
        "14446920b7c55472de3b172f417fbbeac312090add7356a3c947cfb16911dc95",
        "png",
    ),
    (
        "song_sparrow-52",
        "song_sparrow",
        392544542,
        221653008,
        "emckenziewhat",
        138683,
        "7fd2f3ad3c835e541a7c720372d662a8f953f88f4d8223864a03e4b0f70a42d3",
        "jpeg",
    ),
    (
        "song_sparrow-53",
        "song_sparrow",
        270614122,
        156495271,
        "radkins21",
        92783,
        "7bf8ceb5944de93cbf01aebe4d464b9acb848569c07fbe6e732b701d309f0b25",
        "jpg",
    ),
    (
        "song_sparrow-54",
        "song_sparrow",
        139634990,
        84920252,
        "raffib128",
        56293,
        "865cf92a61d89cb3420c9d3cbf585a0724df2e314918b05d705804aee356f1a5",
        "jpeg",
    ),
    (
        "song_sparrow-55",
        "song_sparrow",
        25251350,
        16733911,
        "kemper",
        54034,
        "99f19fa1a37d23712538e43f6a82339f1fd37defe283de3baacc17719f05857e",
        "jpeg",
    ),
    (
        "song_sparrow-56",
        "song_sparrow",
        139129879,
        84639025,
        "wewantashrubbery",
        104736,
        "aabbffe279d90ffdf737175cc720bea92d634483b830c90acecf920901419643",
        "jpg",
    ),
    (
        "song_sparrow-57",
        "song_sparrow",
        128242284,
        78367372,
        "stevestevens",
        112226,
        "918dfc8094d0a697beb1dcd04499cfd7b0c4b12f9dac4d915d1566ab5690b29b",
        "jpeg",
    ),
    (
        "song_sparrow-58",
        "song_sparrow",
        478733235,
        266446909,
        "saintaardvark",
        298253,
        "d3c8d3a58b44d28c6af4748e4e4195a9a2af7dc6fe91541429dc44aa3a659d21",
        "jpg",
    ),
    (
        "song_sparrow-59",
        "song_sparrow",
        623006598,
        342147975,
        "ryman56",
        60213,
        "73e18df350a69ce00158f03c7ec4fda8672539e0a7292b75f1dfa22dbec7ee30",
        "jpg",
    ),
    (
        "chipping_sparrow-00",
        "chipping_sparrow",
        198992636,
        117809422,
        "k-simpkins",
        62563,
        "cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf",
        "jpg",
    ),
    (
        "chipping_sparrow-01",
        "chipping_sparrow",
        248210057,
        144599194,
        "w_mark_c",
        193389,
        "497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150",
        "jpg",
    ),
    (
        "chipping_sparrow-02",
        "chipping_sparrow",
        156350853,
        94266719,
        "ellyne",
        142332,
        "6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2",
        "jpeg",
    ),
    (
        "chipping_sparrow-03",
        "chipping_sparrow",
        16128796,
        11327134,
        "reuvenm",
        70532,
        "5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c",
        "jpeg",
    ),
    (
        "chipping_sparrow-04",
        "chipping_sparrow",
        391300648,
        220982684,
        "carterdorscht",
        208567,
        "c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21",
        "jpeg",
    ),
    (
        "chipping_sparrow-05",
        "chipping_sparrow",
        339456083,
        193252042,
        "rawcomposition",
        46329,
        "41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496",
        "jpg",
    ),
    (
        "chipping_sparrow-06",
        "chipping_sparrow",
        40457652,
        26076708,
        "andywilson",
        156894,
        "b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6",
        "jpeg",
    ),
    (
        "chipping_sparrow-07",
        "chipping_sparrow",
        84151914,
        52921135,
        "davidfbird",
        145714,
        "ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9",
        "jpeg",
    ),
    (
        "chipping_sparrow-08",
        "chipping_sparrow",
        523674610,
        291074747,
        "rwp84",
        47983,
        "41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800",
        "jpg",
    ),
    (
        "chipping_sparrow-09",
        "chipping_sparrow",
        292695018,
        168861389,
        "tim_kirsten",
        64812,
        "cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d",
        "jpeg",
    ),
    (
        "chipping_sparrow-10",
        "chipping_sparrow",
        92501489,
        57964052,
        "tniernberger",
        78380,
        "44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5",
        "jpg",
    ),
    (
        "chipping_sparrow-11",
        "chipping_sparrow",
        220257388,
        129611350,
        "gcart043",
        129377,
        "1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361",
        "jpeg",
    ),
    (
        "chipping_sparrow-12",
        "chipping_sparrow",
        478480946,
        266314208,
        "russnamitz",
        61359,
        "8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b",
        "jpeg",
    ),
    (
        "chipping_sparrow-13",
        "chipping_sparrow",
        538480223,
        298914072,
        "hiltonward",
        201271,
        "2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346",
        "jpg",
    ),
    (
        "chipping_sparrow-14",
        "chipping_sparrow",
        58192408,
        36778771,
        "bradenjudson",
        22352,
        "7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75",
        "jpeg",
    ),
    (
        "chipping_sparrow-15",
        "chipping_sparrow",
        80410971,
        50636049,
        "radrat",
        55646,
        "0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667",
        "jpg",
    ),
    (
        "chipping_sparrow-16",
        "chipping_sparrow",
        300376697,
        172991803,
        "matthias55",
        81986,
        "7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b",
        "jpeg",
    ),
    (
        "chipping_sparrow-17",
        "chipping_sparrow",
        370917657,
        209626382,
        "craigmartin",
        101736,
        "feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5",
        "jpg",
    ),
    (
        "chipping_sparrow-18",
        "chipping_sparrow",
        264119989,
        152960401,
        "laurelthrone",
        194107,
        "13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e",
        "jpeg",
    ),
    (
        "chipping_sparrow-19",
        "chipping_sparrow",
        220323755,
        129631264,
        "enspring",
        85687,
        "886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1",
        "jpeg",
    ),
    (
        "chipping_sparrow-20",
        "chipping_sparrow",
        210319996,
        124105091,
        "bunnymom20",
        188328,
        "2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899",
        "jpeg",
    ),
    (
        "chipping_sparrow-21",
        "chipping_sparrow",
        117013164,
        71832078,
        "terrimewbornagain",
        49527,
        "4456adb52cdafdefdaeb6dd8cec5955f93c1b8f40abdbb1182b2f605a0e1d486",
        "jpeg",
    ),
    (
        "chipping_sparrow-22",
        "chipping_sparrow",
        660951395,
        362143224,
        "mike_cove",
        148079,
        "41d85eaa47b3a41a97ada54728a84c0bfbfed3bcff92de3b6ba88d4c9aface48",
        "jpg",
    ),
    (
        "chipping_sparrow-23",
        "chipping_sparrow",
        691160217,
        377927837,
        "ben142",
        235922,
        "f8137e51e90f0a1964c64fac1a775beaa9ae96c90a1d9aad4f1c870f143e34fd",
        "jpg",
    ),
    (
        "chipping_sparrow-24",
        "chipping_sparrow",
        707633397,
        386482138,
        "leannestacy",
        238800,
        "3fef93babe26eed119ab3b2363902007c46a58a02f91beb0d3bc650821f0dc28",
        "jpg",
    ),
    (
        "chipping_sparrow-25",
        "chipping_sparrow",
        696064427,
        380468188,
        "rdnwoods",
        309547,
        "b56f8f043112021367cbb02861021ac2ef6d2831359315f49d1445bf92477a07",
        "jpg",
    ),
    (
        "chipping_sparrow-26",
        "chipping_sparrow",
        702557361,
        383835541,
        "seanwashington1",
        159095,
        "2940c235b38f84101504b6f50e59c359d0f13e3dc384cadbd43976d1e5e9de91",
        "jpg",
    ),
    (
        "chipping_sparrow-27",
        "chipping_sparrow",
        660337945,
        361814902,
        "kristinpiston",
        158651,
        "a2272e4168edf260001ace410774fbe874ea6b1f6632b8502efc1bc02ae0f866",
        "jpg",
    ),
    (
        "chipping_sparrow-28",
        "chipping_sparrow",
        546647394,
        303298450,
        "kzoebel",
        86457,
        "c37a4aff4263de570ca82f94de92f82d9f9711922e5880547af836104f1883df",
        "jpg",
    ),
    (
        "chipping_sparrow-29",
        "chipping_sparrow",
        713495256,
        389515472,
        "lina47741",
        287986,
        "52e8a37dca8d985a57fdd0d90205e1d813388d961dbf4280d5cbccca9c6af5d3",
        "jpg",
    ),
    (
        "chipping_sparrow-30",
        "chipping_sparrow",
        269951645,
        156136769,
        "conhawn",
        106540,
        "11b48d8bf97084d9ff57aa69517b61539f03b72525022930192330fcb05a160c",
        "jpeg",
    ),
    (
        "chipping_sparrow-31",
        "chipping_sparrow",
        535714979,
        297462154,
        "dissectedfrog",
        200498,
        "cd2f2e1be8c76240a1eeff9535e9a60b8f4fb9d771c7e487d65cde48b7e24cf6",
        "jpg",
    ),
    (
        "chipping_sparrow-32",
        "chipping_sparrow",
        557301829,
        308874987,
        "mar_y_sierra_silvestre",
        115174,
        "ce58e87584df623a86643705b08750228903705ca9b02d93b0206401a24eee9e",
        "jpg",
    ),
    (
        "chipping_sparrow-33",
        "chipping_sparrow",
        652401709,
        357680572,
        "dianeclark6280",
        46782,
        "b77c0eea6899d49f8ab97fd1ad6fc772b6055e5195628fa9f6f989e2d51cedd1",
        "jpg",
    ),
    (
        "chipping_sparrow-34",
        "chipping_sparrow",
        657945260,
        360569392,
        "jasonleduc",
        36363,
        "e3471104742fa68b9eeb149255b0136c53306df23d14f2b2633cea80c9e46eea",
        "jpg",
    ),
    (
        "chipping_sparrow-35",
        "chipping_sparrow",
        434553038,
        243681621,
        "stariplativky",
        57489,
        "e253bd9d70dea9d0438b795f6bebcbc2b26c48b16a90729f0d3155695fa812fd",
        "jpg",
    ),
    (
        "chipping_sparrow-36",
        "chipping_sparrow",
        207498409,
        122551474,
        "askalotl",
        98446,
        "d84e28017be8142c22758e59f14b8c52990e79584a68e6f929123aab4997ac68",
        "jpeg",
    ),
    (
        "chipping_sparrow-37",
        "chipping_sparrow",
        179570776,
        106840941,
        "artemis224",
        138087,
        "74cad843089d2aada7a124f9e704ae1c1419b7cfcade7e2ae29e2c4eda0a6d36",
        "jpg",
    ),
    (
        "chipping_sparrow-38",
        "chipping_sparrow",
        81287617,
        51172999,
        "thyg",
        55659,
        "e1f075addb4bd1feea3ab256e37945a464d2e8c77768b7be131d0655382268d7",
        "jpg",
    ),
    (
        "chipping_sparrow-39",
        "chipping_sparrow",
        343116928,
        195090930,
        "umamimomma",
        52983,
        "a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2",
        "jpg",
    ),
    (
        "chipping_sparrow-40",
        "chipping_sparrow",
        480169680,
        267207764,
        "cvharris",
        144339,
        "d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a",
        "jpeg",
    ),
    (
        "chipping_sparrow-41",
        "chipping_sparrow",
        352953833,
        200088501,
        "aster-asti",
        105786,
        "82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db",
        "jpg",
    ),
    (
        "chipping_sparrow-42",
        "chipping_sparrow",
        622149696,
        341721858,
        "wilderbombyx",
        293652,
        "227c39a25941a81e9551b5ccef39f95d320e7daa02c96de247123ba62f7b9824",
        "jpg",
    ),
    (
        "chipping_sparrow-43",
        "chipping_sparrow",
        503555630,
        280377982,
        "jtdavis05",
        42352,
        "24ccb474659505cd785502295f385ec369a36c6248296aff85261ed3475a4372",
        "jpeg",
    ),
    (
        "chipping_sparrow-44",
        "chipping_sparrow",
        509135437,
        283351340,
        "martyndrabik",
        125948,
        "1604bf4f5512207c59637a84c78842285c40f76a4cf5335bf6d39afe896ed03c",
        "jpg",
    ),
    (
        "chipping_sparrow-45",
        "chipping_sparrow",
        8076820,
        6402608,
        "msieges",
        166588,
        "6c379f94141a0bcdeb8c9743ea140ded50f8c7864baecd6cef0ee1c724b9fd80",
        "jpeg",
    ),
    (
        "chipping_sparrow-46",
        "chipping_sparrow",
        131917390,
        80486493,
        "tom_lazar",
        137355,
        "10ab0a9c5af2ca2203a18e4e3df799dcf129494d7b637a3c46ffdcf0f3b3856c",
        "jpeg",
    ),
    (
        "chipping_sparrow-47",
        "chipping_sparrow",
        141348323,
        85865218,
        "j-dehoog",
        71473,
        "d3fae9ca7f515c43c216d749f085231d19ddfd101bd37cc4ffae25fc946a34fb",
        "jpeg",
    ),
    (
        "chipping_sparrow-48",
        "chipping_sparrow",
        28879424,
        18850970,
        "andy71",
        194983,
        "ea5af97ddef4f743596c176653c86e457f6bddde65d86f5ab16ddff939574683",
        "jpeg",
    ),
    (
        "chipping_sparrow-49",
        "chipping_sparrow",
        148831027,
        90084486,
        "ian-wolfe",
        181942,
        "cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49",
        "jpg",
    ),
    (
        "chipping_sparrow-50",
        "chipping_sparrow",
        641370647,
        351455126,
        "hrachski",
        130262,
        "9bc25019bf6a71d73daac63b810d18039d360887504014525598cffce2035908",
        "jpg",
    ),
    (
        "chipping_sparrow-51",
        "chipping_sparrow",
        648268506,
        355462611,
        "potatocthulhu",
        193138,
        "f526135f7b71e0f9f8d80c17d8e6c1a3cd89ba068333b9104e66d02ced7a7eaa",
        "jpg",
    ),
    (
        "chipping_sparrow-52",
        "chipping_sparrow",
        645412478,
        353758550,
        "ctlqh",
        154844,
        "e34228baa058ddb18c6aacf193c20574dab67d8c1f4d6940eec6e1acb03d993b",
        "jpg",
    ),
    (
        "chipping_sparrow-53",
        "chipping_sparrow",
        121777317,
        74504842,
        "hickl",
        48576,
        "a7f0980bf777cacb9f61855d157b94d895f6dd629485b2a038ce20519c53856c",
        "jpg",
    ),
    (
        "chipping_sparrow-54",
        "chipping_sparrow",
        147992761,
        89606238,
        "benkeen",
        461732,
        "8b6e8f03650c270624ebf82f85dd0b37db3e693157d4d3d7a1787c855846da9d",
        "png",
    ),
    (
        "chipping_sparrow-55",
        "chipping_sparrow",
        27976676,
        18310312,
        "schoenitz",
        66424,
        "64a0186d15abaf1537a34bd329b67625f7580b7aa544dded6bc626dd389b72fe",
        "jpg",
    ),
    (
        "chipping_sparrow-56",
        "chipping_sparrow",
        190706448,
        112884695,
        "jbeusmans",
        109296,
        "1777945eeb6f916ec6f9c0dc10c81014e8dc0ba22d87268e7c7dc1173193b1e6",
        "jpeg",
    ),
    (
        "chipping_sparrow-57",
        "chipping_sparrow",
        184174664,
        109291439,
        "chrismcv",
        91566,
        "c5d9a1dbaf1fad839680c417c7be7d4a680396e38e7c2aba0ee78fd213f7d32c",
        "jpeg",
    ),
    (
        "chipping_sparrow-58",
        "chipping_sparrow",
        72765715,
        45879059,
        "henryfrye",
        146855,
        "f0b957698f5a600e3a6c45471848013f9c74d1602895e40d513e2116a9b7320a",
        "jpg",
    ),
    (
        "chipping_sparrow-59",
        "chipping_sparrow",
        74100878,
        46721618,
        "aredmer",
        132045,
        "ce6f4d4324d684cfc0f40d6fe9c4a2b03a7024080e7448083ae7e3fc0c06ac6e",
        "jpg",
    ),
    (
        "white_throated_sparrow-00",
        "white_throated_sparrow",
        339621218,
        193338380,
        "rawcomposition",
        31152,
        "d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6",
        "jpg",
    ),
    (
        "white_throated_sparrow-01",
        "white_throated_sparrow",
        166821399,
        99992799,
        "dziakj1",
        125954,
        "c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852",
        "jpeg",
    ),
    (
        "white_throated_sparrow-02",
        "white_throated_sparrow",
        469820434,
        261505977,
        "joy4birds",
        111767,
        "7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb",
        "jpg",
    ),
    (
        "white_throated_sparrow-03",
        "white_throated_sparrow",
        260413022,
        150969515,
        "andywilson",
        45958,
        "fb7c533c92239775da6a5b353ff398f6bdd8f65c13445fc1dd72986af5a46466",
        "jpeg",
    ),
    (
        "white_throated_sparrow-04",
        "white_throated_sparrow",
        99351488,
        62040646,
        "bradenjudson",
        23399,
        "e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e",
        "jpeg",
    ),
    (
        "white_throated_sparrow-05",
        "white_throated_sparrow",
        628148203,
        344731686,
        "lavenderdame",
        106872,
        "36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8",
        "jpg",
    ),
    (
        "white_throated_sparrow-06",
        "white_throated_sparrow",
        104660609,
        65043951,
        "allan7",
        42443,
        "10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635",
        "jpeg",
    ),
    (
        "white_throated_sparrow-07",
        "white_throated_sparrow",
        250718938,
        145903421,
        "stevestevens",
        138735,
        "bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-08",
        "white_throated_sparrow",
        15105971,
        10793852,
        "schylerbrown",
        31467,
        "6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-09",
        "white_throated_sparrow",
        171460784,
        102554447,
        "w_mark_c",
        251287,
        "3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137",
        "jpg",
    ),
    (
        "white_throated_sparrow-10",
        "white_throated_sparrow",
        244260317,
        142471526,
        "deejay",
        75694,
        "d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-11",
        "white_throated_sparrow",
        194732675,
        115373159,
        "wildreturn",
        128989,
        "7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831",
        "jpeg",
    ),
    (
        "white_throated_sparrow-12",
        "white_throated_sparrow",
        341990462,
        194541980,
        "ethologist",
        149260,
        "3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb",
        "jpg",
    ),
    (
        "white_throated_sparrow-13",
        "white_throated_sparrow",
        267625208,
        154862375,
        "laurelthrone",
        148616,
        "0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-14",
        "white_throated_sparrow",
        330539586,
        188793537,
        "efalquet",
        119214,
        "c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-15",
        "white_throated_sparrow",
        193091403,
        114346779,
        "ian-wolfe",
        73751,
        "e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479",
        "jpg",
    ),
    (
        "white_throated_sparrow-16",
        "white_throated_sparrow",
        691050773,
        377873006,
        "dinomariobob",
        167744,
        "cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887",
        "jpg",
    ),
    (
        "white_throated_sparrow-17",
        "white_throated_sparrow",
        440986574,
        246671481,
        "suzannehale",
        141857,
        "4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-18",
        "white_throated_sparrow",
        113217820,
        69733707,
        "kemper",
        97802,
        "4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6",
        "jpeg",
    ),
    (
        "white_throated_sparrow-19",
        "white_throated_sparrow",
        575307217,
        318327478,
        "k-simpkins",
        90416,
        "af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b",
        "jpg",
    ),
    (
        "white_throated_sparrow-20",
        "white_throated_sparrow",
        340420076,
        193765555,
        "don54",
        62714,
        "7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556",
        "jpeg",
    ),
    (
        "white_throated_sparrow-21",
        "white_throated_sparrow",
        562344160,
        311510652,
        "rrfc",
        45842,
        "f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0",
        "jpg",
    ),
    (
        "white_throated_sparrow-22",
        "white_throated_sparrow",
        74598266,
        47039878,
        "ianrwhyte",
        141502,
        "b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-23",
        "white_throated_sparrow",
        588001234,
        324825124,
        "portablecity",
        370187,
        "11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607",
        "jpg",
    ),
    (
        "white_throated_sparrow-24",
        "white_throated_sparrow",
        694103481,
        379467387,
        "memoosborne",
        120656,
        "82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a",
        "jpg",
    ),
    (
        "white_throated_sparrow-25",
        "white_throated_sparrow",
        655749108,
        359408087,
        "russnamitz",
        58449,
        "2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1",
        "jpg",
    ),
    (
        "white_throated_sparrow-26",
        "white_throated_sparrow",
        599110899,
        330395670,
        "jd_flores",
        121343,
        "5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae",
        "jpg",
    ),
    (
        "white_throated_sparrow-27",
        "white_throated_sparrow",
        653888173,
        358443775,
        "toknowtheland",
        144197,
        "20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7",
        "jpg",
    ),
    (
        "white_throated_sparrow-28",
        "white_throated_sparrow",
        295037357,
        170132018,
        "sturuss",
        117077,
        "d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3",
        "jpg",
    ),
    (
        "white_throated_sparrow-29",
        "white_throated_sparrow",
        405042885,
        228270930,
        "carterdorscht",
        122157,
        "f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-30",
        "white_throated_sparrow",
        84971412,
        53425343,
        "davidfbird",
        168442,
        "95c5363704d17d6076f3c7a4c9f5e41d235edf7d5dc9d6022116774f2fd544e6",
        "jpg",
    ),
    (
        "white_throated_sparrow-31",
        "white_throated_sparrow",
        476335868,
        265210970,
        "aster-asti",
        91852,
        "cd300cbc823e8ebc0902b7bd80c99eb323e5900bbabf63303b0d9399624c3e54",
        "jpg",
    ),
    (
        "white_throated_sparrow-32",
        "white_throated_sparrow",
        481170792,
        267732071,
        "kcthetc1",
        46021,
        "2194a78339aae99549df56d2d3d7fc91e0bf98a371e7d0da7958b4312fa3b487",
        "jpeg",
    ),
    (
        "white_throated_sparrow-33",
        "white_throated_sparrow",
        623602892,
        342440773,
        "imperialwoodpecker26",
        112678,
        "66653f47ce6a54d38cc1f54b0a85e965f5c5ee7cce65b02c78c1c185cb586c85",
        "jpg",
    ),
    (
        "white_throated_sparrow-34",
        "white_throated_sparrow",
        622732944,
        342010560,
        "tom_lazar",
        34257,
        "584070c4b5231db49087d1507aabec8f51adc4e09dcdda8b93b3a78159d47146",
        "jpg",
    ),
    (
        "white_throated_sparrow-35",
        "white_throated_sparrow",
        505347090,
        281327144,
        "erikschiff",
        85652,
        "91d50bda6afc3cd5596bf4d3176999bf473c31fb6680b64445ec1cbeaabea847",
        "jpg",
    ),
    (
        "white_throated_sparrow-36",
        "white_throated_sparrow",
        7072170,
        5706496,
        "jmutter",
        171582,
        "de90ea2739e8c65ca89f0e511abaf44219501e18377eb49d9e5c7f10d3a6d5b2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-37",
        "white_throated_sparrow",
        16303076,
        11431253,
        "robw",
        61348,
        "e7d81cfe027c7b0914c79a0431f0ce1431db1ebe78368a2fe5f31e2aab59c3c7",
        "jpg",
    ),
    (
        "white_throated_sparrow-38",
        "white_throated_sparrow",
        7149657,
        5761350,
        "bradleysaul",
        108304,
        "adf755b8393fb6b4db99e788d4112c2c72197c33ecffeec2ca7b3ba5dd9d650e",
        "jpg",
    ),
    (
        "white_throated_sparrow-39",
        "white_throated_sparrow",
        27513220,
        17998912,
        "mefisher",
        49587,
        "f1716d49782f70455ec97ecdda634505b6b2afc375105b1d779353e704dda07b",
        "jpg",
    ),
    (
        "white_throated_sparrow-40",
        "white_throated_sparrow",
        642121459,
        351843522,
        "jeffcherry",
        53348,
        "c9e82d7e47ca2e51c856f9bd4d8f2965d8f69948f081c08c9fa6ebf44fcde711",
        "jpg",
    ),
    (
        "white_throated_sparrow-41",
        "white_throated_sparrow",
        642355950,
        351962427,
        "robinlanark",
        54504,
        "c0da16d5146afac102fa274fffea2e0b203282078ed692f3b7ad4a1340b0d4cd",
        "jpg",
    ),
    (
        "white_throated_sparrow-42",
        "white_throated_sparrow",
        638199267,
        349826931,
        "wildaboutwildlife",
        32981,
        "845f7d38d6f35d2e6e9dca61234c9566778fec1e274500c97e47977fcecd232f",
        "jpg",
    ),
    (
        "white_throated_sparrow-43",
        "white_throated_sparrow",
        119839965,
        73407944,
        "terrimewbornagain",
        61951,
        "dbfbaaa2f72caa812c026122f41512caef00f8f8fdd21c2307fcdf9d79921757",
        "jpeg",
    ),
    (
        "white_throated_sparrow-44",
        "white_throated_sparrow",
        6236370,
        5078910,
        "ruggedbynature",
        126403,
        "3d6030593bf57bc5734fd6c0706c1e350bb117a32370a16b194ef4488668f123",
        "jpeg",
    ),
    (
        "white_throated_sparrow-45",
        "white_throated_sparrow",
        32221828,
        20882923,
        "christinan",
        132485,
        "b92ce41128bb1c5b53a0c487b46ba80eea83b9a443ee9c72f100005e5d80e411",
        "jpeg",
    ),
    (
        "white_throated_sparrow-46",
        "white_throated_sparrow",
        36598789,
        23643589,
        "jessm-c",
        45861,
        "eda75171590418596e372c6cba058364acb64f1279d250dff6c5df61fa17868b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-47",
        "white_throated_sparrow",
        163198257,
        98065830,
        "ellyne",
        109261,
        "864fb42a82561684c1ac22ae71bd023ddeb15c6052b7732ebd2570a98b552a1a",
        "jpeg",
    ),
    (
        "white_throated_sparrow-48",
        "white_throated_sparrow",
        162150240,
        97493574,
        "c_burns802",
        188464,
        "0fb2fb2e04bca82c436315b77a974adc665d2d1fb4e419dd61afa068ab412254",
        "jpeg",
    ),
    (
        "white_throated_sparrow-49",
        "white_throated_sparrow",
        161756778,
        97282492,
        "raffib128",
        67237,
        "77384a65eb120bdb345c3015cb8c22fcec9c89df49b0e0c00954e3476834da9f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-50",
        "white_throated_sparrow",
        165146429,
        99107203,
        "philippthompson",
        84518,
        "8582544ca8b3cb1c73174fc42a9d66971fdae0c90cd4739083eb4ef4983eb6e7",
        "jpeg",
    ),
    (
        "white_throated_sparrow-51",
        "white_throated_sparrow",
        177451999,
        105722646,
        "natepow",
        44983,
        "c81a68c46264d1a5fcae3042e7ab24653c3615665cef6293c31622d5f342d840",
        "jpg",
    ),
    (
        "white_throated_sparrow-52",
        "white_throated_sparrow",
        67630852,
        42591255,
        "schoenitz",
        138616,
        "34594c49a4f601b17e49dfc4d4098db7b73ff5a69d3830b773378e4d695ff181",
        "jpg",
    ),
    (
        "white_throated_sparrow-53",
        "white_throated_sparrow",
        99106006,
        61898962,
        "glennberry",
        122576,
        "fd8cd1c0b0c1fdd7636b8588c86c6d4f3cddc233e9bd4bccade5e2ed275d0e28",
        "jpg",
    ),
    (
        "white_throated_sparrow-54",
        "white_throated_sparrow",
        102162606,
        63634844,
        "eug302",
        73329,
        "692114df14fb8a75fdf38428a196467a4a394e5ded95605fbb48cf64903ccb1b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-55",
        "white_throated_sparrow",
        370403033,
        209307218,
        "kuykenwil",
        124312,
        "d0700068b06ad0f79eeac331a92e5d633bd76be264c3d8801a9a6eb301193969",
        "jpg",
    ),
    (
        "white_throated_sparrow-56",
        "white_throated_sparrow",
        500790892,
        278893388,
        "quillipede",
        186273,
        "0db918123c7f529ca76c04129886eb9aeb947bd3e90cdfd968c4605931c16f85",
        "jpeg",
    ),
    (
        "white_throated_sparrow-57",
        "white_throated_sparrow",
        364738073,
        205369541,
        "sandra1142",
        144424,
        "4cd383751871426904f4d8ff7526a312ba33a6c894faf7121c6ec0ff6ce0d7b1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-58",
        "white_throated_sparrow",
        374502828,
        211884837,
        "halliefromcali",
        62476,
        "0073652fddd7378dc936f280c18d7e44c4a7160aadc2f051e7b5e2896b1a476c",
        "jpg",
    ),
    (
        "white_throated_sparrow-59",
        "white_throated_sparrow",
        109241547,
        67591707,
        "blkillin",
        130154,
        "e5b8c1caa0f7cc7926d46eac42df582ccb0882b4a0733fc0740b8b4597a4e014",
        "jpeg",
    ),
    (
        "dark_eyed_junco-00",
        "dark_eyed_junco",
        172110799,
        102901486,
        "schylerbrown",
        182973,
        "185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9",
        "jpeg",
    ),
    (
        "dark_eyed_junco-01",
        "dark_eyed_junco",
        46691943,
        29901256,
        "haida_gwaii",
        46823,
        "a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5",
        "jpg",
    ),
    (
        "dark_eyed_junco-02",
        "dark_eyed_junco",
        707222551,
        386266764,
        "ben142",
        289608,
        "7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89",
        "jpg",
    ),
    (
        "dark_eyed_junco-03",
        "dark_eyed_junco",
        346777340,
        196961623,
        "zacharyfoster",
        46712,
        "ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab",
        "jpg",
    ),
    (
        "dark_eyed_junco-04",
        "dark_eyed_junco",
        8793471,
        6892999,
        "truthseqr",
        45302,
        "f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2",
        "jpeg",
    ),
    (
        "dark_eyed_junco-05",
        "dark_eyed_junco",
        192557376,
        114006980,
        "k-simpkins",
        172398,
        "206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479",
        "jpeg",
    ),
    (
        "dark_eyed_junco-06",
        "dark_eyed_junco",
        243303909,
        141959574,
        "andywilson",
        71321,
        "ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6",
        "jpeg",
    ),
    (
        "dark_eyed_junco-07",
        "dark_eyed_junco",
        274980085,
        159160633,
        "andy71",
        51209,
        "c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b",
        "jpeg",
    ),
    (
        "dark_eyed_junco-08",
        "dark_eyed_junco",
        213798376,
        126031618,
        "nathanael15",
        57646,
        "d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b",
        "jpg",
    ),
    (
        "dark_eyed_junco-09",
        "dark_eyed_junco",
        63482066,
        39977347,
        "chrisleearm",
        41870,
        "90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72",
        "jpeg",
    ),
    (
        "dark_eyed_junco-10",
        "dark_eyed_junco",
        458710472,
        255914048,
        "joy4birds",
        90973,
        "fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153",
        "jpg",
    ),
    (
        "dark_eyed_junco-11",
        "dark_eyed_junco",
        20784016,
        14046286,
        "gambolingquail",
        93957,
        "a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-12",
        "dark_eyed_junco",
        332842159,
        189933284,
        "jan-konilu",
        110145,
        "d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd",
        "jpeg",
    ),
    (
        "dark_eyed_junco-13",
        "dark_eyed_junco",
        513004508,
        270404136,
        "thevertebratepokedex",
        141252,
        "854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f",
        "jpg",
    ),
    (
        "dark_eyed_junco-14",
        "dark_eyed_junco",
        593499256,
        327583635,
        "orionid",
        108583,
        "7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d",
        "jpg",
    ),
    (
        "dark_eyed_junco-15",
        "dark_eyed_junco",
        256948665,
        149140989,
        "igor322",
        86925,
        "9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a",
        "jpeg",
    ),
    (
        "dark_eyed_junco-16",
        "dark_eyed_junco",
        12533281,
        9255403,
        "braincellsgone",
        40857,
        "be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe",
        "jpg",
    ),
    (
        "dark_eyed_junco-17",
        "dark_eyed_junco",
        12580989,
        9282523,
        "artemis224",
        216253,
        "15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093",
        "jpg",
    ),
    (
        "dark_eyed_junco-18",
        "dark_eyed_junco",
        63590391,
        40041059,
        "bobbyblackmore",
        82853,
        "87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-19",
        "dark_eyed_junco",
        106718005,
        66230973,
        "vicki936",
        41475,
        "e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be",
        "jpeg",
    ),
    (
        "dark_eyed_junco-20",
        "dark_eyed_junco",
        611049594,
        336277421,
        "skylar_schell",
        15425,
        "7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e",
        "jpg",
    ),
    (
        "dark_eyed_junco-21",
        "dark_eyed_junco",
        591147962,
        326403963,
        "toknowtheland",
        106373,
        "07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39",
        "jpg",
    ),
    (
        "dark_eyed_junco-22",
        "dark_eyed_junco",
        459696953,
        256398190,
        "w_mark_c",
        262126,
        "5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c",
        "jpg",
    ),
    (
        "dark_eyed_junco-23",
        "dark_eyed_junco",
        484171775,
        269292238,
        "aschuman",
        59459,
        "89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282",
        "jpg",
    ),
    (
        "dark_eyed_junco-24",
        "dark_eyed_junco",
        469746672,
        261469406,
        "shannon_j",
        74167,
        "ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18",
        "jpg",
    ),
    (
        "dark_eyed_junco-25",
        "dark_eyed_junco",
        357382685,
        202362003,
        "dougbrown",
        56324,
        "70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78",
        "jpeg",
    ),
    (
        "dark_eyed_junco-26",
        "dark_eyed_junco",
        7660371,
        6119391,
        "jeffreyleeisanaturalist",
        108394,
        "0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42",
        "jpg",
    ),
    (
        "dark_eyed_junco-27",
        "dark_eyed_junco",
        437957476,
        245430473,
        "eug302",
        106520,
        "394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf",
        "jpeg",
    ),
    (
        "dark_eyed_junco-28",
        "dark_eyed_junco",
        339439778,
        193239701,
        "rawcomposition",
        32258,
        "73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a",
        "jpg",
    ),
    (
        "dark_eyed_junco-29",
        "dark_eyed_junco",
        634100291,
        347744522,
        "zorthesosen",
        169264,
        "932237001fe68e3d6d19c496fd448b8a82f254316211ac6f9d35708a1395ed81",
        "jpg",
    ),
    (
        "dark_eyed_junco-30",
        "dark_eyed_junco",
        6198359,
        5055484,
        "glmory",
        108168,
        "d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965",
        "jpg",
    ),
    (
        "dark_eyed_junco-31",
        "dark_eyed_junco",
        6485959,
        5243663,
        "fake_id",
        148254,
        "ce4dab6c5b37b7ebac07f69d4040d4686fea9fd4b753f91bdf15a3a440820712",
        "jpg",
    ),
    (
        "dark_eyed_junco-32",
        "dark_eyed_junco",
        246318772,
        143610209,
        "wildreturn",
        62090,
        "839e57443659659522ed65dfd054a8fc0a6bfe400518d9f9a7519383d33a2c0d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-33",
        "dark_eyed_junco",
        253199424,
        147174876,
        "sean579",
        121660,
        "6efc17eb4c0b184d64a605a76d3734d9bff5d26e8c8609d2f0b0dd4ccc8b3faa",
        "jpeg",
    ),
    (
        "dark_eyed_junco-34",
        "dark_eyed_junco",
        244315013,
        142500114,
        "enspring",
        63514,
        "1d5012c2b4504b4347fef36d8764eebd1bf07649a14cf874e6d0b3eba9fb3c4f",
        "jpg",
    ),
    (
        "dark_eyed_junco-35",
        "dark_eyed_junco",
        256191686,
        148740024,
        "nartb",
        194532,
        "e88b1449a989fc62c5da1c592e4987fa4db71eadf1ef29c0c5c1159b8f77a298",
        "jpg",
    ),
    (
        "dark_eyed_junco-36",
        "dark_eyed_junco",
        241013731,
        140770209,
        "nana10",
        196479,
        "fc773ca1b06e0e3673aac5c2213c66981ede11bf3ad4a0cdef2102c4f1d99843",
        "jpeg",
    ),
    (
        "dark_eyed_junco-37",
        "dark_eyed_junco",
        240712071,
        140608440,
        "saintaardvark",
        129995,
        "94c4dc30ca31804d6645a6be802cf87ccaffe974f0d3cb554b4fb3e9696ec6a5",
        "jpg",
    ),
    (
        "dark_eyed_junco-38",
        "dark_eyed_junco",
        242909841,
        141750179,
        "calinsdad",
        61773,
        "c49c5d22969552de0fc04ff95eefb8a2ffc76df528477b93c2e2c38cdebe8f6c",
        "jpg",
    ),
    (
        "dark_eyed_junco-39",
        "dark_eyed_junco",
        109385584,
        67667584,
        "ninetoes",
        64185,
        "64e9ee29cf017c292e1711b4c6482488a74c8f652e88e8b7c1df883dc91d75e2",
        "jpg",
    ),
    (
        "dark_eyed_junco-40",
        "dark_eyed_junco",
        102027986,
        63561089,
        "michaelnaumoff",
        114346,
        "eccaa3c60fe575e28fc60f8f6896a99065bff4aee0a4fdc057d7220e7870ca4d",
        "jpg",
    ),
    (
        "dark_eyed_junco-41",
        "dark_eyed_junco",
        105652410,
        65622040,
        "don54",
        159856,
        "5c4282e5e85af7a886eceeadc90977bcd11bf26658176ba5ef8182d2f4a2bfe1",
        "jpeg",
    ),
    (
        "dark_eyed_junco-42",
        "dark_eyed_junco",
        267851501,
        154986181,
        "shanebustapbj",
        203657,
        "217002a366779db0ccc3fddffa71e79472b1fcaa5d444049971dac4c31c910f5",
        "jpeg",
    ),
    (
        "dark_eyed_junco-43",
        "dark_eyed_junco",
        268597047,
        155389350,
        "j-dehoog",
        54916,
        "c0890844944f3d603f8220b0a75213051766c6a45b2ba1fed51381134832f69f",
        "jpeg",
    ),
    (
        "dark_eyed_junco-44",
        "dark_eyed_junco",
        515881506,
        286951393,
        "jubileej",
        108221,
        "9cbecae915f1c96dcc31d5fd77cb5902181152cc3b918693191cc11fce18822c",
        "jpg",
    ),
    (
        "dark_eyed_junco-45",
        "dark_eyed_junco",
        640462423,
        351001488,
        "russnamitz",
        44508,
        "63276eb90201f890b977a54b95bf1d65592eef98089dbb73f2e27e66aaf49a65",
        "jpg",
    ),
    (
        "dark_eyed_junco-46",
        "dark_eyed_junco",
        649025513,
        355899297,
        "rocksand2134",
        233215,
        "2f2545fa5e7901c0383578dea84131eb4be551b8f2bd55d9f308ebc838144f39",
        "jpg",
    ),
    (
        "dark_eyed_junco-47",
        "dark_eyed_junco",
        625101518,
        343192310,
        "jtdavis05",
        106387,
        "4642076ebc5e9bcde85dbba65eb16a49d3fabe6c82cc707b4863ad4b7a4531ae",
        "jpg",
    ),
    (
        "dark_eyed_junco-48",
        "dark_eyed_junco",
        509902416,
        283764048,
        "loarie",
        271588,
        "0c1e3897caca8ff3ff4f3eff7548e30ca50dfe2c2c972106303f9e21c259dbda",
        "jpg",
    ),
    (
        "dark_eyed_junco-49",
        "dark_eyed_junco",
        627787858,
        344552039,
        "margohj",
        54717,
        "a41e557077525b064fa16a7c0b140d608d323663519cfbf424f4454d79951244",
        "jpg",
    ),
    (
        "dark_eyed_junco-50",
        "dark_eyed_junco",
        630444682,
        345899876,
        "robinlanark",
        102578,
        "aa4ffd5dbd553d60ba687d57b21b52ce21572d3c711471abbc5e959bd1126da2",
        "jpg",
    ),
    (
        "dark_eyed_junco-51",
        "dark_eyed_junco",
        635784743,
        348597723,
        "lina47741",
        317439,
        "392d1114b7fc1bb3b041d9895ceb049432a96951764e8949d5939315b96bcc54",
        "jpg",
    ),
    (
        "dark_eyed_junco-52",
        "dark_eyed_junco",
        518407567,
        288278997,
        "zzphantom",
        231733,
        "de2a5be329b583911babe0758c9ce8c0982191f743b8082eb826623256709f71",
        "jpg",
    ),
    (
        "dark_eyed_junco-53",
        "dark_eyed_junco",
        620711072,
        341026776,
        "ethologist",
        186675,
        "13f47961b2a1654ec40f6d1a2005afbf34c75060987fbb662001e66e10a023ed",
        "jpg",
    ),
    (
        "dark_eyed_junco-54",
        "dark_eyed_junco",
        638143588,
        349800424,
        "tom_lazar",
        157629,
        "0be0b84bbc956c616ce35e9e704e8c9e01f0d24db7eeb157fcaa3b3952bd9b23",
        "jpg",
    ),
    (
        "dark_eyed_junco-55",
        "dark_eyed_junco",
        668612910,
        366183670,
        "gendereuphorbia",
        76372,
        "4bfa9f780e9fd0327afec60a8b42e3378b8a9917a0461fd77a6df73b1a94e6bb",
        "jpg",
    ),
    (
        "dark_eyed_junco-56",
        "dark_eyed_junco",
        116078128,
        71308227,
        "msieges",
        47042,
        "ecd4ade5abf058ac887fabe2706eb021ada3b9ab083d10728c9b1dc26de56bbc",
        "jpeg",
    ),
    (
        "dark_eyed_junco-57",
        "dark_eyed_junco",
        117580446,
        72145931,
        "hewwowhy",
        67000,
        "1a8ca3be627c192459440c2eb07bd85b348f0e876a7de32bfc6fe0fb2959a618",
        "jpg",
    ),
    (
        "dark_eyed_junco-58",
        "dark_eyed_junco",
        117044036,
        71848977,
        "radrat",
        81035,
        "7362dd0a27d11b18d5a79abcce9a80304dc80dc0ea1fa84265a1cf0638687641",
        "jpeg",
    ),
    (
        "dark_eyed_junco-59",
        "dark_eyed_junco",
        11110817,
        8359397,
        "giselle9",
        71741,
        "b2e74f27ddca8d1e6dd4807251b07867cbfec8b6bf51d93ef7f142109f705520",
        "jpeg",
    ),
    (
        "house_finch-00",
        "house_finch",
        697940852,
        381438133,
        "ben142",
        196735,
        "a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5",
        "jpg",
    ),
    (
        "house_finch-01",
        "house_finch",
        176982307,
        105476125,
        "vicki936",
        22211,
        "c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f",
        "jpeg",
    ),
    (
        "house_finch-02",
        "house_finch",
        117990649,
        72375345,
        "kristen163",
        75945,
        "eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f",
        "jpeg",
    ),
    (
        "house_finch-03",
        "house_finch",
        389479656,
        220010434,
        "aster-asti",
        82128,
        "a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5",
        "jpg",
    ),
    (
        "house_finch-04",
        "house_finch",
        98576538,
        61594129,
        "enspring",
        46784,
        "8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c",
        "jpg",
    ),
    (
        "house_finch-05",
        "house_finch",
        72470599,
        45698380,
        "henrya",
        61724,
        "1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b",
        "jpeg",
    ),
    (
        "house_finch-06",
        "house_finch",
        80751781,
        50842166,
        "leahmfulton",
        44269,
        "6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7",
        "jpg",
    ),
    (
        "house_finch-07",
        "house_finch",
        214612538,
        126483167,
        "hamiltonturner",
        124116,
        "6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930",
        "jpeg",
    ),
    (
        "house_finch-08",
        "house_finch",
        630196420,
        345777550,
        "truthseqr",
        149455,
        "abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c",
        "jpg",
    ),
    (
        "house_finch-09",
        "house_finch",
        213077180,
        125637342,
        "jnicat",
        25823,
        "e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8",
        "jpeg",
    ),
    (
        "house_finch-10",
        "house_finch",
        500744872,
        278868417,
        "pbaff",
        149626,
        "2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0",
        "jpeg",
    ),
    (
        "house_finch-11",
        "house_finch",
        268678834,
        155431721,
        "stevestevens",
        120325,
        "5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79",
        "jpeg",
    ),
    (
        "house_finch-12",
        "house_finch",
        358373136,
        202884575,
        "kcthetc1",
        52329,
        "b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe",
        "jpeg",
    ),
    (
        "house_finch-13",
        "house_finch",
        104227663,
        64793290,
        "verdantpulsar",
        101003,
        "90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5",
        "jpeg",
    ),
    (
        "house_finch-14",
        "house_finch",
        196156834,
        116218899,
        "kgarrett",
        56801,
        "ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0",
        "jpeg",
    ),
    (
        "house_finch-15",
        "house_finch",
        454332148,
        253709711,
        "rlaortiz",
        149985,
        "cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d",
        "jpeg",
    ),
    (
        "house_finch-16",
        "house_finch",
        665287910,
        295246528,
        "dinomariobob",
        74972,
        "5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49",
        "jpg",
    ),
    (
        "house_finch-17",
        "house_finch",
        509969159,
        283798927,
        "damienxw",
        104277,
        "56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805",
        "jpg",
    ),
    (
        "house_finch-18",
        "house_finch",
        168402062,
        100871757,
        "k-simpkins",
        86922,
        "209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d",
        "jpg",
    ),
    (
        "house_finch-19",
        "house_finch",
        661042217,
        362189851,
        "dougbrown",
        56426,
        "ad704994fda99779aa340ca1c637b1371f0513df6f36973263bb9ee88cf6713b",
        "jpg",
    ),
    (
        "house_finch-20",
        "house_finch",
        701862058,
        383472210,
        "dblanco",
        25222,
        "8562ed9c4e889b34f83b40c55dd93b821a6b5829a3b1c681374a53c3ca9bf01f",
        "jpg",
    ),
    (
        "house_finch-21",
        "house_finch",
        709783562,
        387605980,
        "fraskar",
        99555,
        "60a1926358c4ec04aeec9414381177600580c8237899c852e03472e53cf87f31",
        "jpg",
    ),
    (
        "house_finch-22",
        "house_finch",
        703924789,
        384540425,
        "meshhy",
        110802,
        "2df38d1aaab2c268c7a47f05e55bef1e83690eeeb91f34874c5b479a5ff0a2cb",
        "jpg",
    ),
    (
        "house_finch-23",
        "house_finch",
        705516095,
        385380385,
        "m_aniket",
        77797,
        "ba009764b66ed89efc9b0aaacd9565830c65039967961488d774ce3c9add5d64",
        "jpg",
    ),
    (
        "house_finch-24",
        "house_finch",
        577527110,
        319485540,
        "glmory",
        94407,
        "6d4cd80f8ef77f41575befae0d8534e9bc397685dceb7f629b5cbc24447a3613",
        "jpg",
    ),
    (
        "house_finch-25",
        "house_finch",
        570686433,
        315887836,
        "emilyheaton",
        229263,
        "27e0bd21c1b322aa5d74c5504dbf27bfb5a0fdeea9718b6ffcee7926f2cb2733",
        "jpg",
    ),
    (
        "house_finch-26",
        "house_finch",
        266259557,
        154114419,
        "andywilson",
        47792,
        "b0dc2de7134cc5af50b1482681b450d32b1f58765990d5d553c4838ae7936db0",
        "jpeg",
    ),
    (
        "house_finch-27",
        "house_finch",
        269729570,
        156011712,
        "kbkash",
        59048,
        "3af407624c46ba4218d24a198ed22b812d4967380bd0e7f67e960f7fbb2defe7",
        "jpeg",
    ),
    (
        "house_finch-28",
        "house_finch",
        270277531,
        156183453,
        "aparrot1",
        83424,
        "93c92a1bb7cf89e7106b0c3df65ce030c82bccade181d661952e940ddcc6282a",
        "jpg",
    ),
    (
        "house_finch-29",
        "house_finch",
        658061500,
        360629741,
        "mike_cove",
        107594,
        "deaaefaf712059b6c8a929a83e1ffd0295e12f0f291bc9869011703eecfcf372",
        "jpg",
    ),
    (
        "house_finch-30",
        "house_finch",
        545518752,
        302693775,
        "cvharris",
        208917,
        "fc5ffdbd30bc76fa4371a99f59543c9137411ddac0a691e2e8122ae7796e038b",
        "jpg",
    ),
    (
        "house_finch-31",
        "house_finch",
        662587006,
        362998939,
        "curran",
        113662,
        "644931afff18c942930717ac3ca563b63c18a0c78ab12ff88557aeede18ead63",
        "jpg",
    ),
    (
        "house_finch-32",
        "house_finch",
        665549166,
        364566546,
        "sealgyu",
        130087,
        "e6fb3e42979ce2709433e8141c4c665a70c4942166153578c7f0173328ffb2ad",
        "jpg",
    ),
    (
        "house_finch-33",
        "house_finch",
        566433127,
        313653086,
        "fake_id",
        150986,
        "cb8a50f48f65795850f57c4d8f84e49debc3deed59b6a60d5bee8ef95299650f",
        "jpg",
    ),
    (
        "house_finch-34",
        "house_finch",
        404592864,
        228025269,
        "logan_artz",
        78971,
        "d0cf631b138d199951d96538903eeb1b56e3b8da3f28c7b042d8868060f27a44",
        "jpeg",
    ),
    (
        "house_finch-35",
        "house_finch",
        423118612,
        237726292,
        "conhawn",
        153347,
        "826ddd6fa8fad5bd1450c0c7beb77fa3261f41a0afa61d7681b7096bc4225c40",
        "jpeg",
    ),
    (
        "house_finch-36",
        "house_finch",
        290080883,
        167451466,
        "vijaybarve",
        89129,
        "f0b32b4af4bdf0bc7eaf298b37a9be6dbcb175d9f72553efc239b6afd2746a69",
        "jpeg",
    ),
    (
        "house_finch-37",
        "house_finch",
        296347840,
        170825854,
        "susanaber",
        96563,
        "47832989fc212f19631e55b63c23155817209c40c66af49e4e03e42863243770",
        "jpeg",
    ),
    (
        "house_finch-38",
        "house_finch",
        414288752,
        233136904,
        "wafflemaster135",
        29548,
        "9f2039a678a69e8f8f383d43f80e3f4e6fcc81d5b99285a0e5b5e54834cf9125",
        "jpeg",
    ),
    (
        "house_finch-39",
        "house_finch",
        284751592,
        164572867,
        "efalquet",
        80806,
        "b50e0a0a4a0bae79b5a8cd034645252044601a29b081f0842443848e802e82d2",
        "jpeg",
    ),
    (
        "house_finch-40",
        "house_finch",
        205914234,
        121669780,
        "darylnolan",
        44763,
        "c04e8d7c002ad0038dcb62843e776ffb8fc86f425c2399442dd1653bbaa4b904",
        "jpeg",
    ),
    (
        "house_finch-41",
        "house_finch",
        96688995,
        60467105,
        "erikschiff",
        145431,
        "dd056b1c0d89dcb9fa06b1c38936142e5c825eb2aeed0d7d2265dc7e147be792",
        "jpg",
    ),
    (
        "house_finch-42",
        "house_finch",
        732358790,
        399184296,
        "katrinamccollough",
        41331,
        "76c899fd9b9aa84ad41888c71a36ab99431fe7201dd417f2c19a66f77388420d",
        "jpg",
    ),
    (
        "house_finch-43",
        "house_finch",
        83824778,
        52720178,
        "beesbirdsbugs",
        48816,
        "72731eb63fa9c5058f9911d11be929af9a58cfea805159d54010e9095948c39a",
        "jpg",
    ),
    (
        "house_finch-44",
        "house_finch",
        349400403,
        198261102,
        "paulgraham",
        105965,
        "b0d1f556ee4926e6a6f87201f6b73955dd9c530788e0ce57d5e09ced74ede001",
        "jpeg",
    ),
    (
        "house_finch-45",
        "house_finch",
        484652323,
        269544283,
        "robinlanark",
        29863,
        "a1d6e8ffbcef10a83a2d3674c7228af89f8d39297f44cd569d1e031da02bc516",
        "jpeg",
    ),
    (
        "house_finch-46",
        "house_finch",
        341357046,
        194221751,
        "haconra",
        80643,
        "8e11937a5c6d5a47da47f562d52d4089d58ad3f2adefadc287b30c562c7ece85",
        "jpg",
    ),
    (
        "house_finch-47",
        "house_finch",
        622140195,
        341714166,
        "rachel141",
        133938,
        "aa776ef9fd3022b6b3db824a762bdde82eeb95518a8764763bd8199683bf1d04",
        "jpg",
    ),
    (
        "house_finch-48",
        "house_finch",
        515309527,
        286632225,
        "solunasilver",
        29507,
        "72a6b495ed99de80504c6b1a86ac1fbfc9e5736bc6f43274bd6093cc91f9ea05",
        "jpg",
    ),
    (
        "house_finch-49",
        "house_finch",
        514877896,
        286416757,
        "jubileej",
        153667,
        "26cd596525a244d2ba82ec2d575834c15671080814400cf7862f614c0db68d07",
        "jpg",
    ),
    (
        "house_finch-50",
        "house_finch",
        125605349,
        76831730,
        "sarahangulo",
        60174,
        "236eb345f1cf1ab4c82fb3b2cc48b948e614f8ddfd958c87a62ce5b44a04f080",
        "jpeg",
    ),
    (
        "house_finch-51",
        "house_finch",
        12881677,
        9467101,
        "artemis224",
        173956,
        "91b5f9f66e8b84cd96169894024ebc3519de6f737d3a32ee3b70100c22b039b9",
        "jpg",
    ),
    (
        "house_finch-52",
        "house_finch",
        124405788,
        76034586,
        "radrat",
        36008,
        "bb9b3fe0b37a1ab6342b1e573f3faf3a287669b70fa9f00b5bfce83de4a376c6",
        "jpeg",
    ),
    (
        "house_finch-53",
        "house_finch",
        17468099,
        12163814,
        "andy71",
        199644,
        "6bc02aaf56c1549dd78d6711f9121d48c33fbd76a8fb623895369f05604ad30d",
        "jpg",
    ),
    (
        "house_finch-54",
        "house_finch",
        134699271,
        82117365,
        "seanwashington1",
        35334,
        "d6e0fe4a8cc690fd6369deff8ea67f4810e1f1297a6f1b9f125aae905742db2a",
        "jpeg",
    ),
    (
        "house_finch-55",
        "house_finch",
        134287410,
        81854554,
        "omcelroy",
        90394,
        "6b4a5ea4c8aa1722332ccdf420af0ca0906cfdb87348e9f6c0904d5a92d8c203",
        "jpg",
    ),
    (
        "house_finch-56",
        "house_finch",
        17763955,
        12337763,
        "jennifer510",
        167368,
        "35355c603049fdf21fc3bf1fed827b50740cfdba134542707026bb4e1f32149b",
        "jpeg",
    ),
    (
        "house_finch-57",
        "house_finch",
        18946521,
        13021113,
        "deejay",
        60576,
        "2420363f6b02433f4b03f3323148ed5e4ac9d37c58a27e859f8e48354db866c8",
        "jpeg",
    ),
    (
        "house_finch-58",
        "house_finch",
        14590841,
        10492098,
        "silvercat",
        120515,
        "23934e0376c70dd6d5ce5e9a24c4176291c877432d25257f07e27694495c879a",
        "jpeg",
    ),
    (
        "house_finch-59",
        "house_finch",
        124702263,
        76228853,
        "janeyair",
        162431,
        "5c86d7cbd28e09d44814735f1e0a21e8266bd4e6a4396b047d092d2f5f3f2111",
        "jpeg",
    ),
    (
        "american_goldfinch-00",
        "american_goldfinch",
        84579952,
        53187208,
        "glennberry",
        59673,
        "72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36",
        "jpeg",
    ),
    (
        "american_goldfinch-01",
        "american_goldfinch",
        12533322,
        9255418,
        "braincellsgone",
        55661,
        "6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc",
        "jpg",
    ),
    (
        "american_goldfinch-02",
        "american_goldfinch",
        131102823,
        80016788,
        "radrat",
        98990,
        "232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff",
        "jpeg",
    ),
    (
        "american_goldfinch-03",
        "american_goldfinch",
        175048222,
        104466897,
        "eug302",
        44231,
        "11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e",
        "jpg",
    ),
    (
        "american_goldfinch-04",
        "american_goldfinch",
        68849595,
        43390778,
        "mefisher",
        154503,
        "66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1",
        "jpg",
    ),
    (
        "american_goldfinch-05",
        "american_goldfinch",
        431916465,
        242278180,
        "k-simpkins",
        45413,
        "d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d",
        "jpg",
    ),
    (
        "american_goldfinch-06",
        "american_goldfinch",
        230801384,
        135330401,
        "enspring",
        42886,
        "78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed",
        "jpeg",
    ),
    (
        "american_goldfinch-07",
        "american_goldfinch",
        377136648,
        213398931,
        "nathan1177",
        66516,
        "7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438",
        "jpg",
    ),
    (
        "american_goldfinch-08",
        "american_goldfinch",
        294667312,
        169935316,
        "dande",
        163470,
        "39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0",
        "jpeg",
    ),
    (
        "american_goldfinch-09",
        "american_goldfinch",
        660472044,
        361884286,
        "ben142",
        270599,
        "733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876",
        "jpg",
    ),
    (
        "american_goldfinch-10",
        "american_goldfinch",
        143215717,
        86889530,
        "memoosborne",
        129438,
        "ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba",
        "jpeg",
    ),
    (
        "american_goldfinch-11",
        "american_goldfinch",
        403873164,
        227647158,
        "drew_baxter",
        106009,
        "74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49",
        "jpeg",
    ),
    (
        "american_goldfinch-12",
        "american_goldfinch",
        481153019,
        267722510,
        "vicki936",
        255397,
        "3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36",
        "jpg",
    ),
    (
        "american_goldfinch-13",
        "american_goldfinch",
        213311122,
        125763980,
        "hickl",
        24740,
        "0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd",
        "jpeg",
    ),
    (
        "american_goldfinch-14",
        "american_goldfinch",
        417352824,
        234736320,
        "joy4birds",
        81109,
        "357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd",
        "jpeg",
    ),
    (
        "american_goldfinch-15",
        "american_goldfinch",
        72130720,
        45486482,
        "dctphoto",
        178111,
        "17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509",
        "jpeg",
    ),
    (
        "american_goldfinch-16",
        "american_goldfinch",
        45148898,
        28952026,
        "megachile",
        49358,
        "6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252",
        "jpeg",
    ),
    (
        "american_goldfinch-17",
        "american_goldfinch",
        133251099,
        81250513,
        "raffib128",
        128110,
        "863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef",
        "jpeg",
    ),
    (
        "american_goldfinch-18",
        "american_goldfinch",
        155797129,
        93957328,
        "nathanael15",
        74086,
        "7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5",
        "jpg",
    ),
    (
        "american_goldfinch-19",
        "american_goldfinch",
        11400438,
        8535447,
        "akneidel",
        26423,
        "4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b",
        "jpg",
    ),
    (
        "american_goldfinch-20",
        "american_goldfinch",
        55990791,
        35505213,
        "conhawn",
        84915,
        "69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb",
        "jpg",
    ),
    (
        "american_goldfinch-21",
        "american_goldfinch",
        145464774,
        88186208,
        "wildreturn",
        68204,
        "565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592",
        "jpg",
    ),
    (
        "american_goldfinch-22",
        "american_goldfinch",
        637247336,
        289067166,
        "dinomariobob",
        142312,
        "01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd",
        "jpg",
    ),
    (
        "american_goldfinch-23",
        "american_goldfinch",
        460988055,
        257029007,
        "eric112",
        60314,
        "0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9",
        "jpg",
    ),
    (
        "american_goldfinch-24",
        "american_goldfinch",
        59072170,
        37280587,
        "bradenjudson",
        23468,
        "827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42",
        "jpeg",
    ),
    (
        "american_goldfinch-25",
        "american_goldfinch",
        66515260,
        41915252,
        "reuvenm",
        58833,
        "90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7",
        "jpeg",
    ),
    (
        "american_goldfinch-26",
        "american_goldfinch",
        109875643,
        67928198,
        "artemis224",
        217909,
        "99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61",
        "jpg",
    ),
    (
        "american_goldfinch-27",
        "american_goldfinch",
        219478221,
        129171341,
        "cgmayers",
        96993,
        "93bdee27bb526b60e0edfa518a923b3c10c41ba57221d6a23f4de15bf4a91600",
        "jpeg",
    ),
    (
        "american_goldfinch-28",
        "american_goldfinch",
        99106353,
        61899131,
        "thyg",
        50544,
        "0272bf89b379c2689247f1dbd6f4b36d7f197898baad91d7a8ebdf1c87f88fbd",
        "jpg",
    ),
    (
        "american_goldfinch-29",
        "american_goldfinch",
        223110598,
        131133187,
        "marissa3",
        89214,
        "6a2bb601ac3a70e80e6ccb33d4a24af43128d1e7afe3e92d42e803234bdf532e",
        "jpg",
    ),
    (
        "american_goldfinch-30",
        "american_goldfinch",
        104649681,
        65037800,
        "kemper",
        113821,
        "f4430daf9fca2cf6deb6b988148a5c2e5ae382af8c5db7e5cd37b26e8fbc9461",
        "jpeg",
    ),
    (
        "american_goldfinch-31",
        "american_goldfinch",
        98669936,
        61655828,
        "cloaca_enthusiast",
        31964,
        "3ddcb2e2e6863ef08a40fae7640e05b36ccec9fab9892ac494f095c4588b746f",
        "jpg",
    ),
    (
        "american_goldfinch-32",
        "american_goldfinch",
        347798463,
        197465311,
        "aster-asti",
        116039,
        "e1236dd9850a412ba68386ffd342d180f582db187b83c1d33cb1e09dd51d0043",
        "jpg",
    ),
    (
        "american_goldfinch-33",
        "american_goldfinch",
        461261879,
        257163676,
        "erikschiff",
        40881,
        "eb0b1ea08fcee160edd008ffc386b69b7b50bb89096c44873b978072c581d826",
        "jpg",
    ),
    (
        "american_goldfinch-34",
        "american_goldfinch",
        343726970,
        195390446,
        "robinlanark",
        53508,
        "8b1f4e496755d1a3c4f567c6a48673998b8480ce7d38e7c52bbd94d7ee9901ed",
        "jpeg",
    ),
    (
        "american_goldfinch-35",
        "american_goldfinch",
        352259924,
        199722919,
        "carterdorscht",
        122243,
        "11edfdb53f892b1123df734cb7300e002171baeef21671a31411ff30d156bdc6",
        "jpeg",
    ),
    (
        "american_goldfinch-36",
        "american_goldfinch",
        148829913,
        90084462,
        "ian-wolfe",
        172110,
        "4f6a493313fc596665eb99f8e303ab0ee980b4932a9957993cd00b03b5360fa4",
        "jpg",
    ),
    (
        "american_goldfinch-37",
        "american_goldfinch",
        276388190,
        159958623,
        "dianeclark6280",
        39786,
        "284ca7c9e814fede2c5a15e427a46d7dd82fa6984c4c862c380b0dddff2f86db",
        "jpg",
    ),
    (
        "american_goldfinch-38",
        "american_goldfinch",
        409778763,
        230765968,
        "squidtk",
        210988,
        "b92ccce145f7a320f25edf92633e0c56aa92e8c83890b4a439ce4b3d669cb1a1",
        "jpeg",
    ),
    (
        "american_goldfinch-39",
        "american_goldfinch",
        382773537,
        216432059,
        "rocksand2134",
        195020,
        "ad33d43d9e5650dd441bbd9adb9cc9854a6a1bcc7088a8802a7fb4ccc1ee8b51",
        "jpeg",
    ),
    (
        "american_goldfinch-40",
        "american_goldfinch",
        290287368,
        167563678,
        "sean579",
        105190,
        "8703f245d4b98476f67775ca0c19b7abd7c5edf756c50b2ff502b3d058c08014",
        "jpeg",
    ),
    (
        "american_goldfinch-41",
        "american_goldfinch",
        423112648,
        237722733,
        "andrew2285",
        55719,
        "10709f5df3f4c509afd260ba51c0db5cee89c7c9076c74e9911e9613303d7754",
        "jpeg",
    ),
    (
        "american_goldfinch-42",
        "american_goldfinch",
        422361237,
        237335568,
        "wildaboutwildlife",
        34855,
        "55b0962389b589037e03378472e65ffa0e7ed0fb7e227a7441106a009f093924",
        "jpg",
    ),
    (
        "american_goldfinch-43",
        "american_goldfinch",
        291016386,
        167956837,
        "samallonthesciencemon",
        128110,
        "489934bec3eb5682067cca11cb210c171198e616bb4920753b48bb0e1480bc32",
        "jpg",
    ),
    (
        "american_goldfinch-44",
        "american_goldfinch",
        419250127,
        235728207,
        "paulgraham",
        39211,
        "be1f9ae51a6c4ee92a543ecfe1654698581d7647a3a0b46859a7cf054d8b0fd9",
        "jpeg",
    ),
    (
        "american_goldfinch-45",
        "american_goldfinch",
        416004544,
        234039610,
        "randv",
        105453,
        "942a87723af8aedc86459479437bc474817f19429806f36d8e1236538448e24f",
        "jpeg",
    ),
    (
        "american_goldfinch-46",
        "american_goldfinch",
        416408092,
        234251817,
        "kmayner",
        143385,
        "55259e21e667b88864e2b88ba8dacc8ac5be0d2606699be35c46d2b17fe67fb2",
        "jpeg",
    ),
    (
        "american_goldfinch-47",
        "american_goldfinch",
        116188790,
        71368917,
        "leahmfulton",
        48440,
        "ae01ef8ba2fb714253543a5e773e4e047e95448f4ff4d1e51bc889a20b14e1bc",
        "jpg",
    ),
    (
        "american_goldfinch-48",
        "american_goldfinch",
        118530525,
        72675993,
        "seanwashington1",
        38505,
        "d688edde61a079907cdf5011a32914432f8f7f35c12d355e308aae29b1988738",
        "jpeg",
    ),
    (
        "american_goldfinch-49",
        "american_goldfinch",
        121016576,
        74071097,
        "geenance",
        110180,
        "fb4d1fef09311949043b3c9edc0d3a85d4c20386346093bb6ea39e1b95f756b9",
        "jpeg",
    ),
    (
        "american_goldfinch-50",
        "american_goldfinch",
        3279608,
        2872951,
        "joed",
        115964,
        "c2f1463bfef4c9b157f7e3ebd33fa37570592a4cdfa276f2b7c673f543cbc3ca",
        "jpeg",
    ),
    (
        "american_goldfinch-51",
        "american_goldfinch",
        118140282,
        72458899,
        "melaniegaddy",
        243713,
        "a3d70061032ac8c665f3d17a9118c8e592d53a3669182b23b502e3573a80d8f8",
        "jpg",
    ),
    (
        "american_goldfinch-52",
        "american_goldfinch",
        7669539,
        6125334,
        "cuihenggang",
        61287,
        "36d9570131caf33f3b5cfa6ab99a26c417e7cd8e38d0c1351d1a2938bb6e9bf9",
        "jpeg",
    ),
    (
        "american_goldfinch-53",
        "american_goldfinch",
        9258865,
        7182023,
        "fake_id",
        151987,
        "131d2a52097c9ab18fad318b62e8213c1bd33acddc72065ec0e404c76220c5ee",
        "jpeg",
    ),
    (
        "american_goldfinch-54",
        "american_goldfinch",
        12222432,
        9054511,
        "myacadianforest",
        107758,
        "69b1b5274d74bb28601afc2758e57a88f8d63743b5410c70284a9f4877c721fb",
        "jpeg",
    ),
    (
        "american_goldfinch-55",
        "american_goldfinch",
        128301260,
        78400716,
        "natepow",
        67416,
        "0c988080a47484802806ef14e0d1fdb9205680eacecedbe24f8bbac2aa2873f9",
        "jpg",
    ),
    (
        "american_goldfinch-56",
        "american_goldfinch",
        129842959,
        79296314,
        "nartb",
        117002,
        "f23b2e68f4c10e2679f0a9fe2f34a894bd2deea9c90c2143462e0aa5a83a3daa",
        "jpg",
    ),
    (
        "american_goldfinch-57",
        "american_goldfinch",
        78410307,
        49351195,
        "chrisleearm",
        108066,
        "8a4b273f1a1bd548c48abcf57b86ed35028cbfb949720180a43f24976a26ac8e",
        "jpg",
    ),
    (
        "american_goldfinch-58",
        "american_goldfinch",
        72300928,
        45598176,
        "henrya",
        68593,
        "a0c2869c330aaea08fba3bfeae99392c5588980b1ec21770c220c744125c3b42",
        "jpeg",
    ),
    (
        "american_goldfinch-59",
        "american_goldfinch",
        194724999,
        115367030,
        "vlatassa",
        159105,
        "c456a23238f8f81f93df7919b92dd3fa0123fa16a875c22ec1f9d0ae6ef17324",
        "jpeg",
    ),
)
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 36, "validation": 8, "test": 16}  # per species; 6 species -> 216 / 48 / 96
MIN_RECORDS = 8
MAX_RECORDS = 20_000
MIN_CLASSES = 2
MAX_CLASSES = 100
MAX_LABEL_CHARS = 64
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_LABEL_RE = re.compile(r"^[A-Za-z0-9_ .:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def photo_url(photo_id: int, ext: str = "jpg") -> str:
    """The served object for a pinned photo; `ext` is its recorded original extension (jpg, jpeg or png,
    either case — the bucket key is case-sensitive)."""
    if ext.lower() not in ("jpg", "jpeg", "png"):
        raise ValueError(f"unsupported photo extension {ext!r}")
    return f"{CORPUS_BASE_URL}{photo_id}/medium.{ext}"


def observation_url(observation_id: int) -> str:
    return f"https://www.inaturalist.org/observations/{observation_id}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:
        local = cache / f"{photo_id}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = photo_url(photo_id, ext)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "dimer-siglip2-tutorial/1.0"}
                )
                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 "
                    f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified photo bytes into `{id, image, label}` records with their provenance."""
    out = []
    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "label": label,
                "scientific_name": SPECIES[label][0],
                "common_name": SPECIES[label][1],
                "inat_photo_id": photo_id,
                "inat_observation_url": observation_url(obs_id),
                "observer": user,
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified draw per species: `sizes` counts per class for train / validation / test."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_label.setdefault(str(record["label"]), []).append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        needed = sum(sizes.values())
        if len(pool) < needed:
            raise ValueError(f"{label}: only {len(pool)} records available, need {needed}")
        cursor = 0
        for name, per_class in sizes.items():
            out[name].extend(pool[cursor : cursor + per_class])
            cursor += per_class
    for name in out:
        rng.shuffle(out[name])
        out[name] = [
            {**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(out[name])
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int, *, require_labels: bool = True) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image/label")
    for key in ("id", "image") + (("label",) if require_labels else ()):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image, label = record["id"], record["image"], record.get("label")
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label_name}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(
            f"{label_name}: image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}"
        )
    if label is None and not require_labels:
        item: dict[str, Any] = {"id": rid, "image": image.convert("RGB")}
    else:
        if not isinstance(label, str) or not _LABEL_RE.match(label.strip()):
            raise ValueError(
                f"{label_name}: label must be a non-empty string of at most {MAX_LABEL_CHARS} plain characters"
            )
        item = {"id": rid, "image": image.convert("RGB"), "label": label.strip()}
    for key in (
        "source_id",
        "observer",
        "inat_photo_id",
        "inat_observation_url",
        "scientific_name",
        "common_name",
    ):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of an image dataset; raises ValueError before any model import. Labels are
    required for the probe (`require_labels=True`) and optional for the reconstruction contract, which
    never reads them; a record that carries a label is always checked."""
    if (
        isinstance(records, Mapping)
        or not isinstance(records, Sequence)
        or isinstance(records, (str, bytes))
    ):
        raise ValueError("records must be a list of {id, image, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    counts: dict[str, int] = {}
    for index, record in enumerate(records):
        item = _check_record(record, index, require_labels=require_labels)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if "label" in item:
            counts[item["label"]] = counts.get(item["label"], 0) + 1
        checked.append(item)
    if require_labels and not MIN_CLASSES <= len(counts) <= MAX_CLASSES:
        raise ValueError(
            f"{len(counts)} distinct labels; {MIN_CLASSES}..{MAX_CLASSES} are required"
        )
    sides = [max(r["image"].size) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "classes": sorted(counts),
        "label_counts": dict(sorted(counts.items())),
        "image_side": {"min": min(sides), "max": max(sides)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def class_names(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The sorted label vocabulary of a dataset (the `classes` a head is trained for)."""
    names = sorted({str(r["label"]) for r in records})
    if len(names) < MIN_CLASSES:
        raise ValueError(f"a dataset needs at least {MIN_CLASSES} distinct labels")
    return names


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same photo matches."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], image_digest(r["image"]), r.get("label")] for r in records]
    return _sha256_bytes(
        json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    )


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def observer_overlap(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, int]:
    """How many observers contributed photos to more than one split (an observation, not an assertion)."""
    seen: dict[str, set[str]] = {}
    for name, records in splits.items():
        for record in records:
            if record.get("observer"):
                seen.setdefault(str(record["observer"]), set()).add(name)
    return {
        "observers": len(seen),
        "in_more_than_one_split": sum(1 for s in seen.values() if len(s) > 1),
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (
        0.0 <= val_fraction < 1.0
        and 0.0 < test_fraction < 1.0
        and val_fraction + test_fraction < 1.0
    ):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_label.setdefault(record["label"], []).append(record)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        n_test = max(1, round(len(pool) * test_fraction))
        n_val = round(len(pool) * val_fraction)
        splits["test"].extend(pool[:n_test])
        splits["validation"].extend(pool[n_test : n_test + n_val])
        splits["train"].extend(pool[n_test + n_val :])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, label}` records from a directory or a zip holding `labels.csv` (columns `id`, `file`,
    `label`) beside the image files; images are decoded, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        table = (source / "labels.csv").read_text(encoding="utf-8")
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "labels.csv" not in members:
            raise ValueError("BYOD zip must contain labels.csv")
        table = archive.read(members["labels.csv"]).decode("utf-8")
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError(
            "BYOD datasets must be a directory or a .zip holding labels.csv and the image files"
        )
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "file", "label"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"labels.csv is missing columns {sorted(missing)}")
    out = []
    for row in rows:
        image = loader(row["file"])
        image.load()
        out.append({"id": row["id"], "image": image.convert("RGB"), "label": row["label"]})
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the labels table of a split (id, file, label, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=["id", "file", "label", "observer", "inat_observation_url"]
        )
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "file": f"{record['inat_photo_id']}.jpg"
                    if record.get("inat_photo_id")
                    else record["id"],
                    "label": record["label"],
                    "observer": record.get("observer", ""),
                    "inat_observation_url": record.get("inat_observation_url", ""),
                }
            )
    return out

**Module 6/6:** `src/vit_mae_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""ViT-MAE base: masked-image reconstruction with the pinned ``facebook/vit-mae-base`` weights, a
reconstruction-supervised evaluation contract, a linear-probe reading of the encoder, and a bounded
continuation of the masked-autoencoding pretraining on a labelled photograph set.

The checkpoint is the full pretraining model (``ViTMAEForPreTraining``: a ViT-B/16 encoder and an
8-layer, 512-wide decoder). Its one user-facing operation is **reconstruction**: a seeded random 75 % of the
196 patches is hidden from the encoder, the decoder predicts every patch's pixels, and the loss is the mean
squared error on the hidden patches — the model's own training objective, which is also the only number it
can be read on without labels. ``embed`` exposes the encoder as a feature extractor (mean of the patch
tokens with nothing hidden, L2-normalised), and ``fit_probe`` / ``classify`` / ``evaluate`` read those
features through a linear head the way the fleet reads self-supervised encoders. ``adapt`` continues the
masked-autoencoding objective on the caller's photographs — the decoder and the last encoder blocks — with
validation-loss epoch selection, and the probe is fitted again afterwards so the effect of that continuation
on the features is measured, not assumed.
"""
# ruff: noqa: E501  -- docstrings and record literals kept on single lines at the fleet width

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from io import BytesIO
from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .metrics import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import load_components, stage_missing_files, verify_snapshot` removed — names are kernel globals defined by the carried modules

ImageInput = str | Path | bytes | Image.Image

MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work
MIN_IMAGE_SIDE = 16
MAX_BATCH = 64  # images per embed() / reconstruct() call
DEFAULT_TRAINABLE_BLOCKS = 2  # the last two encoder blocks + the encoder LayerNorm train beside the whole decoder
PARAMETER_COUNT = 111_907_840
ENCODER_PARAMETERS = 85_798_656
DECODER_PARAMETERS = 26_109_184
PROBE_STEPS = 300
PROBE_LR = 0.01
PROBE_WEIGHT_DECAY = 1e-3
KNN_K = 5
MAX_EVAL_RECORDS = 5_000
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.vit-mae-base.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
GRID = IMAGE_SIZE // PATCH_SIZE

INPUT_SCHEMA: dict[str, Any] = {
    "images": (
        "PIL.Image.Image, raw bytes, or a local path decodable by Pillow; any mode, converted to RGB; "
        "remote URLs are refused"
    ),
    "image_size": (
        f"sides in [{MIN_IMAGE_SIDE}, {MAX_IMAGE_SIDE}] px; the processor resizes every image to {IMAGE_SIZE} x {IMAGE_SIZE} "
        f"(aspect ratio not kept) and normalises with the ImageNet mean / std"
    ),
    "mask_ratio": f"fraction of the {NUM_PATCHES} patches hidden from the encoder, in (0, 1); default {DEFAULT_MASK_RATIO}; the mask is seeded",
    "output": (
        "reconstruct: per image the masked-patch MSE (the model's loss), the visible-patch MSE, the mask, and the "
        "reconstructed image (decoder prediction on hidden patches, the input on visible ones); embed: one "
        f"{HIDDEN_SIZE}-d L2-normalised vector per image; classify (after fit_probe): a score per class"
    ),
    "validation": "size and decodability only; nothing checks what the image shows",
}


def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _coerce_image(value: ImageInput) -> Image.Image:
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, bytes):
        with Image.open(BytesIO(value)) as handle:
            handle.load()
            return handle.convert("RGB")
    text = str(value)
    if text.startswith(("http://", "https://")):
        raise ValueError("remote URLs are not accepted; pass a local path, bytes or a PIL image")
    path = Path(text)
    if not path.is_file():
        raise FileNotFoundError(f"image not found: {path}")
    with Image.open(path) as handle:
        handle.load()
        return handle.convert("RGB")


def _check_size(image: Image.Image, what: str) -> None:
    w, h = image.size
    if min(w, h) < MIN_IMAGE_SIDE or max(w, h) > MAX_IMAGE_SIDE:
        raise ValueError(f"{what}: sides must be in [{MIN_IMAGE_SIDE}, {MAX_IMAGE_SIDE}] px, got {w} x {h}")


def _check_images(images: Sequence[ImageInput] | ImageInput) -> list[Image.Image]:
    if isinstance(images, str | Path | bytes | Image.Image):
        images = [images]
    if not images:
        raise ValueError("images must contain at least one image")
    if len(images) > MAX_BATCH:
        raise ValueError(f"at most {MAX_BATCH} images per call")
    out = []
    for index, image in enumerate(images):
        coerced = _coerce_image(image)
        _check_size(coerced, f"image {index}")
        out.append(coerced)
    return out


def _check_mask_ratio(mask_ratio: float) -> float:
    if isinstance(mask_ratio, bool) or not isinstance(mask_ratio, int | float) or not 0.0 < float(mask_ratio) < 1.0:
        raise ValueError("mask_ratio must be a number in (0, 1)")
    return float(mask_ratio)


def validate_inputs(
    images: Sequence[ImageInput] | ImageInput,
    *,
    mask_ratio: float = DEFAULT_MASK_RATIO,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: the input manifest (schema, per-image observations, verdict), raising exactly what
    `reconstruct` / `embed` would raise."""
    coerced = _check_images(images)
    ratio = _check_mask_ratio(mask_ratio)
    if names is not None and len(names) != len(coerced):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[i] if names else f"image-{i}", "mode": image.mode, "size": list(image.size)} for i, image in enumerate(coerced)],
        "mask_ratio": ratio,
        "hidden_patches": int(NUM_PATCHES * ratio),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def _seeded_noise(seed: int, count: int, generator_device: str = "cpu") -> torch.Tensor:
    """The random argument HF's ViTMAE uses to choose the hidden patches, seeded per image."""
    g = torch.Generator(device=generator_device).manual_seed(int(seed))
    return torch.rand(count, NUM_PATCHES, generator=g)


def _record_seed(record_id: str, seed: int) -> int:
    digest = hashlib.sha256(f"{seed}:{record_id}".encode()).digest()
    return int.from_bytes(digest[:4], "big")


class ViTMAEPipeline:
    def __init__(
        self,
        model: Any,
        processor: Any,
        *,
        device: str | torch.device = "cpu",
        checkpoint_path: Path | str | None = None,
        checkpoint_source: str | None = None,
        manifest_verified: bool = False,
        weight_sha256: str | None = None,
        weight_size_bytes: int | None = None,
    ) -> None:
        self.model = model
        self.processor = processor
        self.device = torch.device(device)
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        self.checkpoint_source = checkpoint_source
        self.manifest_verified = manifest_verified
        self.weight_sha256 = weight_sha256
        self.weight_size_bytes = weight_size_bytes
        self.adapter: dict[str, Any] | None = None
        self.probe: dict[str, Any] | None = None  # {"classes", "weight", "bias", ...} after fit_probe
        if hasattr(self.model, "parameters"):  # injected fakes in the offline tests carry none
            self.model.eval()
            for param in self.model.parameters():
                param.requires_grad_(False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | torch.device | None = None,
        cache_dir: str | Path | None = None,
        weights_path: str | Path | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ViTMAEPipeline:
        """Load the one supported checkpoint (fleet snapshot directory → stage absent entries when allowed →
        verify against the manifest and the pinned digests → load the safetensors file strictly)."""
        if weights_dir is not None:
            if weights_path is not None:
                raise ValueError("pass either weights_dir or weights_path, not both")
            stage_missing_files(weights_dir, allow_download=allow_download)
            verify_snapshot(weights_dir)
            weights_path = weights_dir
        model, processor, target_device, _, metadata = load_components(device=device, cache_dir=cache_dir, weights_path=weights_path, return_metadata=True)
        return cls(
            model,
            processor,
            device=target_device,
            checkpoint_path=metadata.get("checkpoint_path"),
            checkpoint_source=metadata.get("checkpoint_source"),
            manifest_verified=metadata.get("manifest_verified", False),
            weight_sha256=metadata.get("weight_sha256"),
            weight_size_bytes=metadata.get("weight_size_bytes"),
        )

    # ------------------------------------------------------------------ tensors

    def _pixels(self, images: Sequence[Image.Image]) -> torch.Tensor:
        return self.processor(images=list(images), return_tensors="pt")["pixel_values"].to(self.device)

    def _patchify(self, pixels: torch.Tensor) -> torch.Tensor:
        return self.model.patchify(pixels)

    def _forward(self, pixels: torch.Tensor, noise: torch.Tensor, mask_ratio: float) -> Any:
        """One masked forward with the given noise; the mask ratio is set on the config for the call."""
        previous = self.model.config.mask_ratio
        self.model.config.mask_ratio = mask_ratio
        try:
            return self.model(pixel_values=pixels, noise=noise.to(pixels.device))
        finally:
            self.model.config.mask_ratio = previous

    # ------------------------------------------------------------------ inference contract

    def reconstruct(
        self,
        images: Sequence[ImageInput] | ImageInput,
        *,
        mask_ratio: float = DEFAULT_MASK_RATIO,
        seed: int = 0,
        return_images: bool = True,
    ) -> dict[str, Any]:
        """Hide a seeded random `mask_ratio` of the patches, reconstruct, and score against the true pixels."""
        coerced = _check_images(images)
        ratio = _check_mask_ratio(mask_ratio)
        pixels = self._pixels(coerced)
        noise = _seeded_noise(seed, len(coerced))
        with torch.no_grad():
            out = self._forward(pixels, noise, ratio)
            target = self._patchify(pixels)
            pred = out.logits
            mask = out.mask
            filled = pred * mask.unsqueeze(-1) + target * (1 - mask.unsqueeze(-1))
            recon = self.model.unpatchify(filled) if return_images else None
        results = []
        target_np, pred_np, mask_np = target.cpu().numpy(), pred.float().cpu().numpy(), mask.cpu().numpy()
        for i, image in enumerate(coerced):
            masked = masked_mse(pred_np[i], target_np[i], mask_np[i])
            visible = masked_mse(pred_np[i], target_np[i], 1.0 - mask_np[i])
            entry: dict[str, Any] = {
                "input_size": list(image.size),
                "mask_ratio": ratio,
                "hidden_patches": int(mask_np[i].sum()),
                "masked_mse": masked,
                "visible_mse": visible,
                "mask": mask_np[i].astype(bool).tolist(),
            }
            if return_images:
                entry["reconstruction"] = self._to_image(recon[i])
            results.append(entry)
        return {
            "results": results,
            "model_loss": float(out.loss) if out.loss is not None else None,
            "mask_ratio": ratio,
            "seed": int(seed),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_license": MODEL_LICENSE,
        }

    def _to_image(self, pixels: torch.Tensor) -> Image.Image:
        mean = torch.tensor(self.processor.image_mean, dtype=pixels.dtype, device=pixels.device).view(3, 1, 1)
        std = torch.tensor(self.processor.image_std, dtype=pixels.dtype, device=pixels.device).view(3, 1, 1)
        array = ((pixels * std + mean).clamp(0, 1) * 255).round().byte().permute(1, 2, 0).cpu().numpy()
        return Image.fromarray(array)

    def embed(self, images: Sequence[ImageInput] | ImageInput) -> dict[str, Any]:
        """Encoder features with nothing hidden: the mean of the patch tokens after the final LayerNorm, L2-normalised."""
        coerced = _check_images(images)
        with torch.no_grad():
            features = self._features(self._pixels(coerced))
        return {"embeddings": features.cpu().numpy(), "dim": HIDDEN_SIZE, "normalized": True, "pooling": "mean of patch tokens, nothing hidden", "model_id": MODEL_ID, "model_revision": MODEL_REVISION}

    def _features(self, pixels: torch.Tensor) -> torch.Tensor:
        previous = self.model.config.mask_ratio
        self.model.config.mask_ratio = 0.0
        try:
            hidden = self.model.vit(pixel_values=pixels).last_hidden_state
        finally:
            self.model.config.mask_ratio = previous
        pooled = hidden[:, 1:, :].mean(dim=1)
        return torch.nn.functional.normalize(pooled.float(), dim=-1)

    def features(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 16) -> np.ndarray:
        out = []
        for start in range(0, len(records), batch_size):
            chunk = [_coerce_image(r["image"]) for r in records[start : start + batch_size]]
            with torch.no_grad():
                out.append(self._features(self._pixels(chunk)).cpu().numpy())
        return np.concatenate(out, axis=0) if out else np.zeros((0, HIDDEN_SIZE), dtype=np.float32)

    # ------------------------------------------------------------------ reconstruction evaluation

    def evaluate_reconstruction(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        mask_ratio: float = DEFAULT_MASK_RATIO,
        seed: int = 0,
        batch_size: int = 16,
    ) -> dict[str, Any]:
        """Masked-patch MSE over a validated `{id, image, ...}` set with one seeded mask per record id, beside the
        mean-patch and blur fills scored on the same masks."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS, require_labels=False)["records"]
        ratio = _check_mask_ratio(mask_ratio)
        rows, mean_rows, blur_rows = [], [], []
        for start in range(0, len(checked), batch_size):
            chunk = checked[start : start + batch_size]
            pixels = self._pixels([_coerce_image(r["image"]) for r in chunk])
            noise = torch.stack([_seeded_noise(_record_seed(str(r["id"]), seed), 1)[0] for r in chunk])
            with torch.no_grad():
                out = self._forward(pixels, noise, ratio)
                target = self._patchify(pixels).cpu().numpy()
            pred, mask = out.logits.float().cpu().numpy(), out.mask.cpu().numpy()
            for i, record in enumerate(chunk):
                rows.append({"id": str(record["id"]), "masked_mse": masked_mse(pred[i], target[i], mask[i]), "visible_mse": masked_mse(pred[i], target[i], 1.0 - mask[i]), "hidden_patches": int(mask[i].sum())})
                mean_rows.append({"id": str(record["id"]), "masked_mse": masked_mse(mean_patch_fill(target[i], mask[i]), target[i], mask[i]), "visible_mse": 0.0})
                blur_rows.append({"id": str(record["id"]), "masked_mse": masked_mse(blur_fill(target[i], mask[i], grid=GRID), target[i], mask[i]), "visible_mse": 0.0})
        result = reconstruction_metrics(rows, mask_ratio=ratio)
        result["per_image"] = rows
        result["seed"] = int(seed)
        result["baselines"] = {
            "mean_patch_fill": {**reconstruction_metrics(mean_rows, mask_ratio=ratio), "baseline": "every hidden patch = the mean colour of the visible patches"},
            "blur_fill": {**reconstruction_metrics(blur_rows, mask_ratio=ratio), "baseline": "every hidden patch = the mean colour of its visible neighbours"},
        }
        result["verdict"] = "measured" if len(rows) >= MIN_SCORED_RECORDS else "small-sample"
        result["adapted"] = self.adapter is not None
        return result

    # ------------------------------------------------------------------ linear probe

    def fit_probe(
        self,
        train: Sequence[Mapping[str, Any]],
        *,
        steps: int = PROBE_STEPS,
        lr: float = PROBE_LR,
        weight_decay: float = PROBE_WEIGHT_DECAY,
        seed: int = 0,
    ) -> dict[str, Any]:
        """A multinomial logistic-regression head on the frozen, standardised features of a labelled set
        (full-batch Adam, seeded)."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import class_names, validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(train, min_records=MIN_SCORED_RECORDS // 5)["records"]
        classes = class_names(checked)
        gold = torch.tensor([classes.index(str(r["label"])) for r in checked])
        started = time.perf_counter()
        raw = torch.from_numpy(self.features(checked)).float()
        # The MAE paper probes through a BatchNorm without affine parameters; the same standardisation with the
        # training set's statistics, applied to every query, is what makes a linear head on MAE features usable.
        mean, std = raw.mean(dim=0), raw.std(dim=0) + 1e-6
        feats = (raw - mean) / std
        torch.manual_seed(seed)
        head = torch.nn.Linear(HIDDEN_SIZE, len(classes))
        optimiser = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=weight_decay)
        losses = []
        for _ in range(steps):
            optimiser.zero_grad()
            loss = torch.nn.functional.cross_entropy(head(feats), gold)
            loss.backward()
            optimiser.step()
            losses.append(float(loss.detach()))
        self.probe = {
            "classes": classes,
            "weight": head.weight.detach().clone(),
            "bias": head.bias.detach().clone(),
            "feature_mean": mean,
            "feature_std": std,
            "steps": steps,
            "lr": lr,
            "weight_decay": weight_decay,
            "n_train": len(checked),
            "train_loss": [losses[0], losses[-1]],
            "standardisation": "training-set mean / std per feature (the paper's affine-free BatchNorm)",
            "seconds": round(time.perf_counter() - started, 2),
            "train_features": raw.numpy(),
            "train_gold": gold.numpy(),
        }
        return {k: v for k, v in self.probe.items() if k not in ("weight", "bias", "feature_mean", "feature_std", "train_features", "train_gold")}

    def _scores(self, features: np.ndarray) -> np.ndarray:
        if self.probe is None:
            raise ValueError("no probe fitted: call fit_probe() first")
        feats = (torch.from_numpy(np.asarray(features)).float() - self.probe["feature_mean"]) / self.probe["feature_std"]
        logits = feats @ self.probe["weight"].T + self.probe["bias"]
        return torch.softmax(logits, dim=-1).numpy()

    def classify(self, images: Sequence[ImageInput] | ImageInput) -> dict[str, Any]:
        """Scores per class from the fitted probe on the encoder features."""
        coerced = _check_images(images)
        with torch.no_grad():
            feats = self._features(self._pixels(coerced)).cpu().numpy()
        scores = self._scores(feats)
        classes = self.probe["classes"]
        return {
            "classes": list(classes),
            "results": [{"top1": classes[int(row.argmax())], "scores": {c: float(row[j]) for j, c in enumerate(classes)}} for row in scores],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]], *, k: int = KNN_K) -> dict[str, Any]:
        """Probe accuracy / macro F1 on a validated labelled set, with a k-NN on the same features beside it."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self.probe is None:
            raise ValueError("no probe fitted: call fit_probe() first")
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        classes = list(self.probe["classes"])
        for record in checked:
            if str(record["label"]) not in classes:
                raise ValueError(f"record label {record['label']!r} is not a probe class")
        gold = [classes.index(str(r["label"])) for r in checked]
        feats = self.features(checked)
        probe = classification_metrics(self._scores(feats), gold, classes)
        if self.probe.get("train_features") is not None:
            knn = classification_metrics(knn_scores(self.probe["train_features"], self.probe["train_gold"], feats, len(classes), k=k), gold, classes)
            knn["baseline"] = f"{k}-nearest-neighbour vote on the same features"
            probe["knn"] = knn
        else:
            probe["knn"] = None  # a reloaded probe carries its head and statistics, not the training features
        probe["classes"] = classes
        probe["predictions"] = [classes[int(row.argmax())] for row in self._scores(feats)]
        probe["verdict"] = "measured" if len(checked) >= MIN_SCORED_RECORDS else "small-sample"
        probe["adapted"] = self.adapter is not None
        return probe

    # ------------------------------------------------------------------ adaptation contract

    def _trainable_names(self, trainable_blocks: int) -> list[str]:
        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int) or not 0 <= trainable_blocks <= ENCODER_LAYERS:
            raise ValueError(f"trainable_blocks must be an int in 0..{ENCODER_LAYERS}")
        first = ENCODER_LAYERS - trainable_blocks
        prefixes = tuple(f"vit.encoder.layer.{k}." for k in range(first, ENCODER_LAYERS)) + ("decoder.",)
        if trainable_blocks:
            prefixes += ("vit.layernorm.",)
        return [name for name, _p in self.model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 5,
        lr: float = 1e-5,
        batch_size: int = 8,
        trainable_blocks: int = DEFAULT_TRAINABLE_BLOCKS,
        mask_ratio: float = DEFAULT_MASK_RATIO,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded continuation of the masked-autoencoding objective on validated photographs.

        Trains the decoder and the last `trainable_blocks` encoder blocks (with the encoder LayerNorm) on the
        checkpoint's own loss — the masked-patch MSE under a fresh random mask every step (`mask_ratio`) — with
        AdamW at a fixed learning rate, gradient clipping at 1.0, seeded order and masks, no scheduler. Epoch 0
        records the frozen model's validation reconstruction; every epoch is scored on the validation set with
        its fixed seeded masks, and the epoch with the lowest validation masked MSE is kept. Labels are not used.
        Transactional: any failure restores the base tensors. The fitted probe (if any) is discarded, because
        the features it was fitted on no longer exist."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        ratio = _check_mask_ratio(mask_ratio)
        names = self._trainable_names(trainable_blocks)
        train_checked = validate_dataset(train, require_labels=False)["records"]
        val_checked = validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS, require_labels=False)["records"] if val else []
        model = self.model
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.05)
        started = time.perf_counter()
        pixels_all = torch.cat([self._pixels([_coerce_image(r["image"])]).cpu() for r in train_checked])

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            result = self.evaluate_reconstruction(val_checked, mask_ratio=ratio, seed=seed)
            return {k: result[k] for k in ("masked_mse", "masked_mse_visible", "psnr_masked", "n")}

        def key(entry: dict[str, Any]) -> float:
            return -entry["val"]["masked_mse"] if entry["val"] else -math.inf

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    chunk = order[start : start + batch_size]
                    pixels = pixels_all[chunk].to(self.device)
                    noise = torch.rand(len(chunk), NUM_PATCHES, generator=generator)
                    optimiser.zero_grad(set_to_none=True)
                    loss = self._forward(pixels, noise, ratio).loss
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": float(np.mean(losses)), "val": score_val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked:
                    if key(entry) > key(history[best_epoch]):
                        best_epoch = epoch
                        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                else:
                    best_epoch = epoch
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
            if progress is not None and history:
                pass
        except BaseException:
            model.load_state_dict({**model.state_dict(), **initial_state}, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            raise
        model.load_state_dict({**model.state_dict(), **best_state}, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.probe = None
        self.adapter = {
            "objective": "masked-autoencoding continuation (masked-patch MSE, the checkpoint's own loss)",
            "mask_ratio": ratio,
            "trainable_blocks": trainable_blocks,
            "n_trainable": int(n_trainable),
            "n_total": int(sum(p.numel() for p in model.parameters())),
            "epochs": epochs,
            "lr": lr,
            "batch_size": batch_size,
            "seed": seed,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "best_epoch": best_epoch,
            "selection": "lowest validation masked MSE (validation split, fixed seeded masks)" if val_checked else "final epoch (no validation split)",
            "seconds": round(time.perf_counter() - started, 2),
            "history": history,
            "trainable_names": names,
        }
        return dict(self.adapter)

    # ------------------------------------------------------------------ artifact

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors (and the fitted probe, if any) as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        probe_record = None
        if self.probe is not None:
            tensors["probe.weight"] = self.probe["weight"].detach().cpu().contiguous()
            tensors["probe.bias"] = self.probe["bias"].detach().cpu().contiguous()
            tensors["probe.feature_mean"] = self.probe["feature_mean"].detach().cpu().contiguous()
            tensors["probe.feature_std"] = self.probe["feature_std"].detach().cpu().contiguous()
            probe_record = {k: v for k, v in self.probe.items() if k not in ("weight", "bias", "feature_mean", "feature_std", "train_features", "train_gold")}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": DEFAULT_MODEL_KEY, "weight_file": MODEL_FILENAME, "weight_sha256": MODEL_SHA256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "probe": probe_record,
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [{"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries (and restore the probe head when the artifact carries one)."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = _check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_blocks"]))
        probe_record = manifest.get("probe")
        if probe_record:
            expected = sorted(expected + ["probe.weight", "probe.bias", "probe.feature_mean", "probe.feature_std"])
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = self.model.state_dict()
        model_tensors = {k: v for k, v in tensors.items() if not k.startswith("probe.")}
        for key, value in model_tensors.items():
            if key not in state or not key.startswith(("vit.encoder.layer.", "vit.layernorm.", "decoder.")):
                raise ValueError(f"artifact tensor {key} is not an adaptable tensor of the base")
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, base has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in model_tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": [k for k in manifest["tensors"] if not k.startswith("probe.")], "history": manifest.get("history", [])}
        self.probe = None
        if probe_record:
            weight, bias = tensors["probe.weight"], tensors["probe.bias"]
            mean, std = tensors["probe.feature_mean"], tensors["probe.feature_std"]
            if tuple(weight.shape) != (len(probe_record["classes"]), HIDDEN_SIZE) or tuple(bias.shape) != (len(probe_record["classes"]),) or tuple(mean.shape) != (HIDDEN_SIZE,) or tuple(std.shape) != (HIDDEN_SIZE,):
                raise ValueError("artifact probe head has an unexpected shape")
            self.probe = {**probe_record, "weight": weight.float(), "bias": bias.float(), "feature_mean": mean.float(), "feature_std": std.float(), "train_features": None, "train_gold": None}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | torch.device | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ViTMAEPipeline:
        """A fresh pipeline from the pinned base with an adapter overlaid."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe


def _check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> Path:
    """Refuse an artifact whose manifest is not exactly the one this package writes. Nothing is deserialised
    here; the digest check that follows detects corruption or drift relative to the adjacent manifest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
        raise ValueError(f"artifact format_version {manifest.get('format_version')!r} is not the supported {ARTIFACT_FORMAT_VERSION!r}")
    base = manifest.get("base_model", {})
    if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (MODEL_ID, MODEL_REVISION, MODEL_SHA256):
        raise ValueError("artifact was adapted from a different base model, revision or weight file")
    if base.get("weight_file", MODEL_FILENAME) != MODEL_FILENAME:
        raise ValueError("artifact was adapted from a different base weight file")
    files = manifest.get("files")
    if not isinstance(files, list) or len(files) != 1:
        raise ValueError("artifact manifest must list exactly one file")
    entry = files[0]
    if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
        raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
    weights_path = (root / entry["path"]).resolve()
    if weights_path.parent != root.resolve():
        raise ValueError("artifact weight path must resolve inside the artifact directory")
    adapter = manifest.get("adapter")
    blocks = adapter.get("trainable_blocks") if isinstance(adapter, Mapping) else None
    if isinstance(blocks, bool) or not isinstance(blocks, int) or not 0 <= blocks <= ENCODER_LAYERS:
        raise ValueError("artifact manifest does not record an in-range integer trainable_blocks")
    probe = manifest.get("probe")
    if probe is not None and (not isinstance(probe, Mapping) or not isinstance(probe.get("classes"), list) or not probe["classes"]):
        raise ValueError("artifact probe record must list its classes")
    if not isinstance(manifest.get("tensors"), list):
        raise ValueError("artifact manifest must list its tensors")
    return weights_path


def load_pipeline(**kwargs: Any) -> ViTMAEPipeline:
    return ViTMAEPipeline.from_pretrained(**kwargs)


def evaluation_report(result: Mapping[str, Any], *, sample_kind: str = "synthetic") -> dict[str, Any]:
    """The per-call reading of a `reconstruct` result: masked and visible MSE per image and their means, with a
    `sample-sanity` verdict on a drawing (plumbing evidence) or `measured` on a scored set."""
    rows = result.get("results", [])
    if not rows:
        raise ValueError("no results to report")
    masked = [float(r["masked_mse"]) for r in rows]
    visible = [float(r["visible_mse"]) for r in rows]
    metrics = [
        {"id": "masked_mse", "value": float(np.mean(masked)), "definition": RECONSTRUCTION_DEFINITIONS["masked_mse"]},
        {"id": "masked_mse_visible", "value": float(np.mean(visible)), "definition": RECONSTRUCTION_DEFINITIONS["masked_mse_visible"]},
        {"id": "mask_ratio", "value": float(result.get("mask_ratio", DEFAULT_MASK_RATIO)), "definition": RECONSTRUCTION_DEFINITIONS["mask_ratio"]},
    ]
    return {
        "metrics": metrics,
        "n": len(rows),
        "sample_kind": sample_kind,
        "verdict": "sample-sanity" if sample_kind.startswith("synthetic") or len(rows) < MIN_SCORED_RECORDS else "measured",
        "note": "a reconstruction loss is the model's own objective, not a quality judgement; lower is better and nothing here is calibrated",
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


__all__ = [
    "ARTIFACT_FORMAT",
    "DEFAULT_TRAINABLE_BLOCKS",
    "INPUT_SCHEMA",
    "MAX_BATCH",
    "MAX_IMAGE_SIDE",
    "MIN_IMAGE_SIDE",
    "PARAMETER_COUNT",
    "ViTMAEPipeline",
    "evaluation_report",
    "load_pipeline",
    "validate_inputs",
]

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `25b184bea553…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ViTMAEPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "vit-mae-base",
  "modelId": "facebook/vit-mae-base",
  "revision": "25b184bea5538bf5c4c852c79d221195fdd2778d",
  "files": [
    {
      "path": "README.md",
      "bytes": 2968,
      "sha256": "16ae81a13455621be2a1e323fcef6824663c011ebd4e75ead22c5b8f13b9d2c5"
    },
    {
      "path": "config.json",
      "bytes": 676,
      "sha256": "1f7a507e1f7f63a262c5500b86470c3b22ff1a9e42de81bbba27105396572dae"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 217,
      "sha256": "a250969c94afba52d785a0e08dd36e13aeda97c4dd2b7fd0d24b457288536cea"
    },
    {
      "path": "model.safetensors",
      "bytes": 447670680,
      "sha256": "479dcef4bd5df06259399027b789f21e9d9a1b79f37155a64176d55bc26fdae8"
    }
  ],
  "totalBytes": 447674541
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ViTMAEPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. iNaturalist photographs and split

`fetch_corpus` returns the 360 pinned photographs from the cache under `weights/inat-birds/` or the iNaturalist open-data bucket — every cached file is re-hashed and every fetched file refused on any byte-size or SHA-256 mismatch — and `read_corpus` decodes them into `{id, image, label}` records with their observation page, observer and species names. `build_sample_dataset` draws a seeded stratified split per species (36 / 8 / 16 → 216 / 48 / 96). `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no photograph (by decoded-pixel digest) is shared, `observer_overlap` reports how many observers contributed to more than one split (an observation about the draw, not an assertion), and the training split's labels table is written to `outputs/vit_mae_pretraining_train.csv` in the shape BYOD expects. The labels are for the probe in Sections 6 and 8; the continuation in Section 7 never reads them.

Look for: 360 photographs, the six species with 36 / 8 / 16 each, three digests, and four refusal probes — a duplicate id, an image over the side ceiling, a dataset with one label and one too small to split — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    display_names = {}
    raw_rows = {'byod': len(records)}
else:
    corpus_files = fetch_corpus(cache_dir='weights/inat-birds')
    corpus = read_corpus(corpus_files)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME}: {CORPUS_RELEASE} ({CORPUS_LICENSE})'
    display_names = {key: common for key, (_scientific, common) in SPECIES.items()}
    raw_rows = {'photographs': len(corpus), 'bytes': sum(len(v) for v in corpus_files.values()), 'observers': len({r['observer'] for r in corpus})}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
classes = class_names(train_records)
write_dataset_csv(train_records, 'outputs/vit_mae_pretraining_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'observer_overlap': observer_overlap(splits), 'classes': classes})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'label_counts': manifest['label_counts'], 'image_side': manifest['image_side'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {'id': example['id'], 'label': example['label'], 'display_name': display_names.get(example['label'], example['label']), 'size': list(example['image'].size), 'observation': example.get('inat_observation_url')}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'image over the side ceiling': [{**train_records[0], 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8))}, *train_records[1:8]],
    'one label only': [{**r, 'label': 'bird'} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Mask and reconstruct through the inference contract

The inference contract is exercised on three 32×32 synthetic shapes — a red square, a green circle and a blue triangle — rendered in code as ASCII PPM files exactly as the repository's `examples/sample-data/generate_samples.py` renders them and digest-asserted against `SHA256SUMS`, a different image family from the photographs, and images the model will be asked to reconstruct again after adaptation. `validate_inputs` applies exactly the checks the public operations apply and returns an input manifest; a remote URL is validated too and its rejection recorded as a finding. `reconstruct` hides a seeded random 75 % of the 196 patches (147 of them), runs the encoder on the visible 49 and the decoder on all 196, and returns per image the **masked MSE** (over the hidden patches, in the processor's normalised pixel space), the **visible MSE** (the decoder also predicts the patches the encoder saw), the mask itself and the reconstruction composited over the visible patches. `embed` returns the mean of the 196 patch tokens with nothing hidden, L2-normalised. The per-call `evaluation_report` on three drawn shapes is `sample-sanity` — plumbing evidence, not a measurement; whether the model reconstructs *well* is what Section 6 measures on 96 photographs.

Look for: the same seed hiding the same 147 patches twice, a different seed hiding different ones, and the pipeline's mean masked MSE equal to the model's own `loss` — the pipeline computes what the checkpoint was trained on. On these flat drawings the hidden patches are trivial to fill (masked MSE about 0.0005–0.02 in the build record) while the visible ones are not (about 0.55–0.64): the decoder was never trained on the patches the encoder saw, which is why the visible-patch error is reported but never read as quality.

In [ ]:
SAMPLE_DIGESTS = {  # examples/sample-data/SHA256SUMS
    'red_square.ppm': 'b38ff0c9131677ed6cf09832eff40a22841e1a4725d426fad3b1bf6a1dbdb096',
    'green_circle.ppm': '2e3e657686a0f6a6df3f3621d21a410d20d9a47b109f06f9405faa4f747a663f',
    'blue_triangle.ppm': 'f0f4c38c7af92b3a6b1d55a25272156edd87b41e029e116cfa059003dae029a3',
}
WIDTH, HEIGHT, BACKGROUND = 32, 32, (245, 245, 245)


def shape_mask(shape, x, y):
    if shape == 'square':
        return 8 <= x < 24 and 8 <= y < 24
    if shape == 'circle':
        return (x - 16) ** 2 + (y - 16) ** 2 <= 9**2
    if not 7 <= y < 26:
        return False
    half = (y - 7) // 2
    return 16 - half <= x <= 16 + half


def render_ppm(foreground, shape):
    # The repository's generate_samples.py rendering: ASCII P3, 24 values per line.
    lines = ['P3', f'{WIDTH} {HEIGHT}', '255']
    for y in range(HEIGHT):
        row = []
        for x in range(WIDTH):
            pixel = foreground if shape_mask(shape, x, y) else BACKGROUND
            row.extend(str(v) for v in pixel)
        for start in range(0, len(row), 24):
            lines.append(' '.join(row[start : start + 24]))
    return '\n'.join(lines) + '\n'


Path('outputs/sample-data').mkdir(parents=True, exist_ok=True)
specs = {'red_square.ppm': ((220, 40, 40), 'square'), 'green_circle.ppm': ((40, 170, 75), 'circle'), 'blue_triangle.ppm': ((40, 90, 220), 'triangle')}
shape_images = []
for name, (foreground, shape) in specs.items():
    path = Path('outputs/sample-data') / name
    path.write_bytes(render_ppm(foreground, shape).encode('ascii'))
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != SAMPLE_DIGESTS[name]:
        raise ValueError(f'Synthetic sample digest mismatch for {name}: {digest} != {SAMPLE_DIGESTS[name]}')
    shape_images.append(path)
shape_sha256 = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in shape_images}
print({'ceilings': {'DEFAULT_MASK_RATIO': DEFAULT_MASK_RATIO, 'NUM_PATCHES': NUM_PATCHES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'MIN_CLASSES': MIN_CLASSES, 'image_contract': '224x224 RGB after processor resize, 196 patches of 16x16', 'device': str(pipe.device)}})
input_manifest = validate_inputs(shape_images, names=[p.name for p in shape_images])
try:
    validate_inputs('https://example.invalid/not-allowed.png')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'remote-url-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/vit_mae_pretraining_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'shapes': [p.name for p in shape_images], 'sha256': {k: v[:16] + '...' for k, v in shape_sha256.items()}, 'manifest_verdict': input_manifest['verdict'], 'hidden_patches': input_manifest['hidden_patches'], 'findings': len(input_manifest['findings'])})
SCENE_SEED = 7
t0 = time.perf_counter()
scene = pipe.reconstruct(shape_images, seed=SCENE_SEED)
again = pipe.reconstruct(shape_images, seed=SCENE_SEED, return_images=False)
other = pipe.reconstruct(shape_images, seed=SCENE_SEED + 1, return_images=False)
shape_embeddings = pipe.embed(shape_images)['embeddings']
scene_rows = []
for path, entry in zip(shape_images, scene['results'], strict=True):
    entry['reconstruction'].save(f'outputs/vit_mae_pretraining_frozen_{path.stem}.png')
    scene_rows.append({'image': path.name, 'hidden_patches': entry['hidden_patches'], 'masked_mse': entry['masked_mse'], 'visible_mse': entry['visible_mse']})
    print(path.name, '->', {'hidden_patches': entry['hidden_patches'], 'masked_mse': round(entry['masked_mse'], 4), 'visible_mse': round(entry['visible_mse'], 4)})
checks = {
    'one_result_per_image': len(scene['results']) == len(shape_images),
    'hidden_patches_match_ratio': all(e['hidden_patches'] == int(NUM_PATCHES * DEFAULT_MASK_RATIO) for e in scene['results']),
    'errors_non_negative': all(e['masked_mse'] >= 0.0 and e['visible_mse'] >= 0.0 for e in scene['results']),
    'same_seed_same_mask': all(a['mask'] == b['mask'] for a, b in zip(scene['results'], again['results'], strict=True)),
    'other_seed_other_mask': any(a['mask'] != b['mask'] for a, b in zip(scene['results'], other['results'], strict=True)),
    'pipeline_mse_is_model_loss': abs(scene['model_loss'] - float(np.mean([e['masked_mse'] for e in scene['results']]))) < 1e-4,
    'embeddings_unit_norm': bool(np.allclose(np.linalg.norm(shape_embeddings, axis=1), 1.0, atol=1e-4)),
}
if not all(checks.values()):
    raise RuntimeError(f'inference output failed a sanity check: {checks}')
frozen_scene = evaluation_report(scene, sample_kind='synthetic')
print({'checks': checks, 'seconds': round(time.perf_counter() - t0, 2), 'frozen_scene': {m['id']: round(m['value'], 4) for m in frozen_scene['metrics']}, 'verdict': frozen_scene['verdict'], 'embedding_shape': list(shape_embeddings.shape)})

## 6. Baselines and the frozen model on the test photographs, two ways

**Reconstruction.** `pipe.evaluate_reconstruction` hides one seeded 75 % mask per photograph (the seed is derived from the record id, so the same photograph gets the same mask in every evaluation of this notebook) and reports the mean and median **masked MSE**, the **visible MSE** and the **PSNR of the hidden patches** over the 96 test photographs. Two non-neural fills are scored on exactly the same masks: the **mean-patch fill** (every hidden patch is the mean colour of the visible patches — what a model that learned nothing about images would do) and the **blur fill** (every hidden patch is the mean colour of its visible neighbours — the cheapest use of locality). Expect the frozen model well below both: the build record measured masked MSE 0.2281 against 0.7652 (mean fill) and 0.5452 (blur fill), PSNR 19.3 dB on the hidden patches.

**Linear probe.** `pipe.fit_probe` embeds the 216 training photographs with nothing hidden, standardises the features with the training set's mean and standard deviation (the paper's affine-free BatchNorm) and fits a multinomial logistic-regression head by full-batch Adam; `pipe.evaluate` reports its **accuracy** and **macro F1** on the test photographs with a per-species breakdown, and beside it a **5-nearest-neighbour** vote on the same features. The **majority floor** answers every photograph with the most frequent training label (accuracy 1/6 on a balanced split), and the **colour nearest neighbour** answers with the label of the training photograph whose 3×3 mean-colour grid is closest. MAE features are known to probe poorly before fine-tuning (the paper reports 67.8 % linear-probe top-1 on ImageNet against 83.6 % after fine-tuning), and six bird species from 216 photographs is a hard probe: the build record measured accuracy 0.365 / macro F1 @P:FROZEN_PROBE_F1@ against the majority floor's 0.167, the colour neighbour's 0.260 and the k-NN's 0.219. Read the per-species recall: it is where the probe's number comes from.

In [ ]:
RK = ('masked_mse', 'masked_mse_median', 'masked_mse_visible', 'psnr_masked')
CK = ('accuracy', 'macro_f1')
t0 = time.perf_counter()
frozen_rec = pipe.evaluate_reconstruction(test_records, seed=0)
frozen_rec_seconds = round(time.perf_counter() - t0, 1)
print({'frozen_reconstruction_test': {k: round(frozen_rec[k], 4) for k in RK}, 'n': frozen_rec['n'], 'mask_ratio': frozen_rec['mask_ratio'], 'verdict': frozen_rec['verdict'], 'seconds': frozen_rec_seconds})
for name, fill in frozen_rec['baselines'].items():
    print({name: {k: round(fill[k], 4) for k in RK}, 'note': fill['baseline']})
print({'definitions': frozen_rec['definitions']})
assert frozen_rec['masked_mse'] < frozen_rec['baselines']['blur_fill']['masked_mse'] < frozen_rec['baselines']['mean_patch_fill']['masked_mse']

baseline_majority = majority_baseline(train_records, test_records, classes)
baseline_neighbour = colour_neighbour_baseline(train_records, test_records, classes)
print({'majority_baseline': {k: round(baseline_majority[k], 3) for k in CK}, 'n': baseline_majority['n'], 'note': baseline_majority['baseline']})
print({'colour_neighbour_baseline': {k: round(baseline_neighbour[k], 3) for k in CK}, 'note': baseline_neighbour['baseline']})
t0 = time.perf_counter()
frozen_probe_fit = pipe.fit_probe(train_records)
frozen_probe = pipe.evaluate(test_records)
probe_seconds = round(time.perf_counter() - t0, 1)
print({'frozen_probe_test': {k: round(frozen_probe[k], 3) for k in CK}, 'n': frozen_probe['n'], 'verdict': frozen_probe['verdict'], 'train_loss': [round(v, 4) for v in frozen_probe_fit['train_loss']], 'standardisation': frozen_probe_fit['standardisation'], 'seconds': probe_seconds})
print({'knn_on_the_same_features': {k: round(frozen_probe['knn'][k], 3) for k in CK}, 'note': frozen_probe['knn']['baseline']})
frozen_fields = {c: {'n': v['support'], 'recall': round(v['recall'], 2), 'f1': round(v['f1'], 2)} for c, v in frozen_probe['per_class'].items()}
print({'by_species_frozen': frozen_fields})
assert frozen_probe['accuracy'] > baseline_majority['accuracy']

## 7. Bounded continuation of the masked-autoencoding objective

`pipe.adapt` continues pre-training: it trains the whole decoder, the last `TRAINABLE_BLOCKS` encoder blocks and the encoder's final LayerNorm — two blocks by default, 40,286,464 of 111,907,840 parameters; the patch embedding, the position embeddings and the first ten blocks stay frozen — on the checkpoint's own loss, the masked-patch MSE under a fresh seeded random 75 % mask every step. AdamW at a fixed learning rate with weight decay 0.05, gradient clipping at 1.0, seeded shuffling and masks, no scheduler, and **no labels**. Epoch 0 records the frozen model's validation reconstruction; every epoch is scored on the 48 validation photographs with their fixed seeded masks, and the epoch with the lowest validation masked MSE is kept — so the selection can return the frozen model itself (epoch 0) when nothing beats it, and never returns a worse one. The fitted probe is discarded, because the features it was fitted on no longer exist.

Watch the numbers move very little: the pre-trained model has seen 1,600 epochs of ImageNet, and 216 photographs of birds are not a new domain to it. The build record's learning-rate sweep found `1e-4` makes the validation masked MSE **worse** from the first epoch (batches of eight with random masks are a noisy gradient), and `1e-5` moves it from 0.2335 to 0.2332 at epoch 3 of 5. That is the honest default: the configuration that did not hurt, chosen on the validation split.

In [ ]:
EPOCHS = 5  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_BLOCKS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 4) for k in ('masked_mse', 'masked_mse_visible', 'psnr_masked')})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=TRAINABLE_BLOCKS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'objective': adapt_result['objective'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})
val_history = {h['epoch']: h['val']['masked_mse'] for h in adapt_result['history']}
assert val_history[adapt_result['best_epoch']] <= val_history[0]  # the selector never returns an epoch worse than the frozen model

## 8. Held-out evaluation, two ways

The test photographs were never used for training or epoch selection, and no photograph appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6: the same seeded masks per photograph for the reconstruction (so a delta is the model's, not the mask's), and a probe re-fitted on the adapted features of the same 216 training photographs with the same seed. The systems are put side by side — three on reconstruction (mean fill, blur fill, frozen, adapted), five on the probe (majority, colour neighbour, k-NN, frozen probe, adapted probe) — the per-species breakdown is repeated and the evaluation report is written as JSON. Read it in this order: the **validation masked MSE** that selected the epoch, then the **test masked MSE** (the build record measured 0.2281 → 0.2280), then the probe (0.365 → 0.365 accuracy, @P:FROZEN_PROBE_F1@ → @P:ADAPTED_PROBE_F1@ macro F1) and the per-species recall. Ninety-six photographs from one seeded split give **no dispersion estimate**; accuracy moves in steps of one photograph; and a masked MSE that moved in the fourth decimal is the honest reading of continued pre-training on a converged model — sample-sanity evidence that the adaptation contract works, not a benchmark, and not a claim about your images until you measure them.

In [ ]:
adapted_rec = pipe.evaluate_reconstruction(test_records, seed=0)
adapted_val_rec = pipe.evaluate_reconstruction(val_records, seed=0)
adapted_probe_fit = pipe.fit_probe(train_records)
adapted_probe = pipe.evaluate(test_records)
adapted_fields = {c: {'n': v['support'], 'recall': round(v['recall'], 2), 'f1': round(v['f1'], 2)} for c, v in adapted_probe['per_class'].items()}
comparison = {
    'reconstruction': {k: {'mean_patch_fill': round(frozen_rec['baselines']['mean_patch_fill'][k], 4), 'blur_fill': round(frozen_rec['baselines']['blur_fill'][k], 4), 'frozen': round(frozen_rec[k], 4), 'adapted': round(adapted_rec[k], 4)} for k in RK},
    'reconstruction_delta_vs_frozen': {k: round(adapted_rec[k] - frozen_rec[k], 5) for k in RK},
    'validation_masked_mse': {'frozen': round(val_history[0], 4), 'selected_epoch': adapt_result['best_epoch'], 'selected': round(val_history[adapt_result['best_epoch']], 4), 'rescored': round(adapted_val_rec['masked_mse'], 4)},
    'probe': {k: {'majority': round(baseline_majority[k], 3), 'colour_neighbour': round(baseline_neighbour[k], 3), 'knn_frozen': round(frozen_probe['knn'][k], 3), 'knn_adapted': round(adapted_probe['knn'][k], 3), 'frozen': round(frozen_probe[k], 3), 'adapted': round(adapted_probe[k], 3)} for k in CK},
    'probe_delta_vs_frozen': {k: round(adapted_probe[k] - frozen_probe[k], 3) for k in CK},
    'by_species': {c: {'n': frozen_fields[c]['n'], 'frozen_recall': frozen_fields[c]['recall'], 'adapted_recall': adapted_fields[c]['recall'], 'frozen_f1': frozen_fields[c]['f1'], 'adapted_f1': adapted_fields[c]['f1']} for c in classes},
}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': DEFAULT_MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'classes': classes,
    'display_names': display_names,
    'reconstruction': {'frozen_test': {k: v for k, v in frozen_rec.items() if k != 'per_image'}, 'adapted_test': {k: v for k, v in adapted_rec.items() if k != 'per_image'}, 'adapted_validation': {k: v for k, v in adapted_val_rec.items() if k != 'per_image'}},
    'probe': {'baselines': {'majority': baseline_majority, 'colour_neighbour': baseline_neighbour}, 'frozen_test': {k: v for k, v in frozen_probe.items() if k != 'predictions'}, 'frozen_fit': frozen_probe_fit, 'adapted_test': {k: v for k, v in adapted_probe.items() if k != 'predictions'}, 'adapted_fit': adapted_probe_fit},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/vit_mae_pretraining_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert abs(adapted_val_rec['masked_mse'] - val_history[adapt_result['best_epoch']]) < 1e-4  # the kept epoch re-scores to the number that selected it
assert adapted_rec['masked_mse'] < adapted_rec['baselines']['blur_fill']['masked_mse']
print({'report': 'outputs/vit_mae_pretraining_evaluation_report.json'})

## 9. Re-reconstruct the drawn shapes, export the adapter and reload it

The three shapes from Section 5 are reconstructed again by the adapted model under the same seeded masks — drawings, a different image family from the photographs it was tuned on, so this is a small look at what the continuation did *outside* its corpus (the build record's numbers are in `docs/release-verification.md`; a changed error here is a finding to record, not a failure) — and reported with the per-call `evaluation_report` (`sample-sanity`). Both readings are written as JSON with the reconstructions as PNG.

`pipe.save_artifact` writes the trained tensors — the decoder, the last two encoder blocks and the encoder LayerNorm, about 161 MB — plus the probe head and its standardisation statistics as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the objective and mask ratio, the probe record (classes, steps, learning rate), the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `ViTMAEPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the declared blocks, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical masked MSE on eight test photographs under the same masks and identical probe decisions (VER4).

In [ ]:
import shutil

adapted_scene = pipe.reconstruct(shape_images, seed=SCENE_SEED)
adapted_scene_rows = []
for path, before, after in zip(shape_images, scene['results'], adapted_scene['results'], strict=True):
    after['reconstruction'].save(f'outputs/vit_mae_pretraining_adapted_{path.stem}.png')
    adapted_scene_rows.append({'image': path.name, 'hidden_patches': after['hidden_patches'], 'masked_mse': after['masked_mse'], 'visible_mse': after['visible_mse']})
    print({'image': path.name, 'frozen_masked_mse': round(before['masked_mse'], 4), 'adapted_masked_mse': round(after['masked_mse'], 4), 'same_mask': before['mask'] == after['mask']})
adapted_scene_report = evaluation_report(adapted_scene, sample_kind='synthetic')
print({'scene_after_adaptation': {m['id']: round(m['value'], 4) for m in adapted_scene_report['metrics']}, 'verdict': adapted_scene_report['verdict']})
with open('outputs/vit_mae_pretraining_shapes.json', 'w', encoding='utf-8') as handle:
    json.dump({'seed': SCENE_SEED, 'frozen': scene_rows, 'adapted': adapted_scene_rows}, handle, indent=2)

artifact_dir = Path('outputs/vit_mae_pretraining_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'vit_mae_pretraining', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'probe_classes': artifact_manifest['probe']['classes'], 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = ViTMAEPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
parity_images = [r['image'] for r in test_records[:8]]
before = pipe.reconstruct(parity_images, seed=11, return_images=False)['results']
after = reloaded.reconstruct(parity_images, seed=11, return_images=False)['results']
before_cls, after_cls = pipe.classify(parity_images)['results'], reloaded.classify(parity_images)['results']
parity = {
    'max_abs_masked_mse_difference': float(max(abs(a['masked_mse'] - b['masked_mse']) for a, b in zip(before, after, strict=True))),
    'identical_reconstructions': int(sum(abs(a['masked_mse'] - b['masked_mse']) < 1e-6 for a, b in zip(before, after, strict=True))),
    'identical_probe_decisions': int(sum(a['top1'] == b['top1'] for a, b in zip(before_cls, after_cls, strict=True))),
    'max_abs_probe_score_difference': float(max(abs(a['scores'][c] - b['scores'][c]) for a, b in zip(before_cls, after_cls, strict=True) for c in a['scores'])),
    'of': len(parity_images),
}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch'], 'reloaded_probe_classes': reloaded.probe['classes']})
assert parity['identical_reconstructions'] == parity['of'] and parity['identical_probe_decisions'] == parity['of'] and parity['max_abs_probe_score_difference'] < 1e-4

write_provenance('outputs/provenance.json', pipeline=pipe)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'fetched_this_run': fetched, 'weight_file': MODEL_FILENAME, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': MODEL_SHA256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'base_url': CORPUS_BASE_URL, 'bytes': CORPUS_BYTES, 'pinned_photographs': len(SAMPLE_RECORDS), 'species': {k: list(v) for k, v in SPECIES.items()}},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'shapes': {'names': [p.name for p in shape_images], 'sha256': shape_sha256, 'seed': SCENE_SEED}, 'frozen_report': frozen_scene, 'adapted_report': adapted_scene_report, 'frozen_rows': scene_rows, 'adapted_rows': adapted_scene_rows},
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'device': str(pipe.device), 'dtype': 'float32', 'checkpoint_source': pipe.checkpoint_source},
}
with open('outputs/vit_mae_pretraining_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model reconstructs hidden patches of photographs it was never trained on far better than the two non-neural fills (masked MSE 0.2281 against 0.5452 and 0.7652 in the build record), and its mean-pooled features carry enough to lift a linear probe over six bird species above the majority floor, the colour neighbour and a k-NN on the same features (0.365 against 0.167, 0.260 and 0.219) — while still being, as the paper says of MAE features, a poor linear-probe backbone until fine-tuned. A bounded continuation of the masked-autoencoding objective on 216 photographs, chosen on the validation split, moves the held-out masked MSE from 0.2281 to 0.2280 and the probe from 0.365 to 0.365, and exports a 161 MB adapter that reloads to identical reconstructions and probe decisions. That is the claim: the adaptation contract works end to end on a real photograph set, the selector is honest enough to keep the frozen model when nothing beats it, and the numbers it produces are read on the objective and on a probe, per species, against non-neural baselines and the frozen model rather than in isolation.

What the numbers say is that continued pre-training on a few hundred in-domain photographs does not change a model that has seen 1,600 epochs of ImageNet: the build record's learning-rate sweep (`docs/release-verification.md`) found `1e-4` hurts from the first epoch and `1e-5` moves the validation masked MSE in the fourth decimal. Where continuation earns its place is a domain the checkpoint has not seen — medical, satellite, microscopy, line art — and that is what BYOD is for; the assertion in Section 7 (the kept epoch is never worse than the frozen model) and the baselines in Sections 6 and 8 are what make such a run readable. The test split is 96 photographs from one seeded draw of one sample, the validation split that picks the epoch is 48, a masked MSE is the model's own objective and not a perceptual judgement, and the probe's accuracy moves in steps of one photograph.

Three things to carry to real data. **Baselines first:** the two fills and the three non-neural classifiers on *your* images are the numbers to read before any adapted one. **Leakage:** keep every photograph in one split (the contract de-duplicates by decoded pixels) and split by photographer or session when your images come from few sources — the sample's observer overlap is printed for exactly that reason. **The readout is not the product:** a linear probe measures the features; a classifier you would ship is a full fine-tuning of the encoder, which this repository does not expose.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real photograph set, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded continuation of the pre-training objective, evaluate against non-neural baselines and the frozen model on an image-disjoint split two ways, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, representation quality on any other population or camera, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_BLOCKS = 0` to train the decoder alone and compare the artifact size; raise `LEARNING_RATE` to `1e-4` and watch the selector keep epoch 0; change `DEFAULT_MASK_RATIO` in `reconstruct` calls (`mask_ratio=0.5`) and read how the masked MSE falls as the encoder sees more; or bring your own images from an unfamiliar domain through BYOD and read the fills before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/vit-mae-pretraining-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/vit-mae-pretraining-pipeline/blob/main/MODEL_CARD.md
- Sample dataset card (synthetic shapes): https://github.com/kurtvalcorza/vit-mae-pretraining-pipeline/blob/main/examples/sample-data/DATASET_CARD.md
- Upstream model: https://huggingface.co/facebook/vit-mae-base
- Upstream code: https://github.com/facebookresearch/mae — served through https://github.com/huggingface/transformers (`ViTMAEForPreTraining`)
- Masked Autoencoders Are Scalable Vision Learners (He, Chen, Xie, Li, Dollár and Girshick, CVPR 2022): https://arxiv.org/abs/2111.06377
- The SigLIP zero-shot row in this fleet, sharing the corpus and the code shape: https://github.com/kurtvalcorza/siglip-v1-zero-shot-pipeline
- iNaturalist open data (CC0 photographs, each observer's own licence): https://www.inaturalist.org/pages/developers — bucket https://inaturalist-open-data.s3.amazonaws.com/
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)